# Análise Quantitativa da Frequência Presencial Semanal do Executivo na Empresa

#### Extra. Calculando um valor somente para dar uma estatística aproximada: Criado pelo Nicholas Birochi

In [ ]:
'''# Calculando os dais úteis entre o período de Janeiro de 2024 até Março de 2025:
# Datas de início e fim do período:
inicio = '2024-01-01'
fim = '2025-03-31'

# Feriados nacionais brasileiros (pode expandir com estaduais/municipais):
feriados = [
    #~~~~~~~~~~~~~~~~~~~~2024~~~~~~~~~~~~~~~~~~~~~#
    '2024-01-01',  # Confraternização Universal...
    '2024-02-12',  # Carnaval...
    '2024-02-13',  # Carnaval...
    '2024-03-29',  # Sexta-feira Santa...
    '2024-04-21',  # Tiradentes...
    '2024-05-01',  # Dia do Trabalho...
    '2024-09-07',  # Independência...
    '2024-10-12',  # Nossa Senhora Aparecida...
    '2024-11-02',  # Finados...
    '2024-11-15',  # Proclamação da República...
    '2024-12-25',  # Natal...

    #~~~~~~~~~~~~~~~~~~~~2025~~~~~~~~~~~~~~~~~~~~~#
    '2025-01-01',  # Confraternização Universal...
    '2025-03-03',  # Carnaval...
    '2025-03-04'   # Carnaval...
]

# Férias Coletivas:
ferias = [
    *pd.date_range('2023-12-18', '2024-01-07', freq='B').strftime('%Y-%m-%d'),
    *pd.date_range('2024-12-23', '2025-01-12', freq='B').strftime('%Y-%m-%d')
]

# Unir feriados e férias:
datas_excluir = sorted(set(feriados + ferias))

# Contar dias úteis excluindo esses dias:
dias_uteis_reais = np.busday_count(inicio, fim, holidays=datas_excluir)

# Visualizando a quantidade de dias úteis:
print(f'Dias úteis reais (excluindo feriados e férias): {dias_uteis_reais}')'''

## 1. Extraindo os Dados:

#### Importando bibliotecas essenciais para a Análise de Dados: Melhorado pelo Nicholas Birochi

In [ ]:
from dash import Dash, html, dcc, Input, Output, State      # Framework para criar dashboards interativos...
from matplotlib.colors import ListedColormap                # Importando as cores dos gráficos...
import matplotlib.patheffects as patheffects                # Alerta visual do "Termômetro-gaauge"...
from dash.exceptions import PreventUpdate                   # Para conseguir minimizar as tabelas detalhadas do Relatório Geral...
from matplotlib.patches import Rectangle                    # Usando para construir o Termômetro...
from datetime import datetime, timedelta                    # Manipulação de datas e intervalos...
import matplotlib.gridspec as gridspec                      # Layouts flexíveis de subplots...
from matplotlib.lines import Line2D                         # Fazer a Linha Pontilhada do gráfico de Tendência...
from collections import defaultdict                         # Permite criar dicionários com valores padrão automáticos...
import plotly.graph_objects as go                           # Criação de gráficos interativos com Plotly...
from dash import callback_context                           # Resetando os filtros no Relatório Geral...
import matplotlib.lines as mlines                           # Fazendo a linha pontilhada do Relatório Específico...
import matplotlib.pyplot as plt                             # Criação de gráficos...
import datetime as dt                                       # Datas e horários (alias)...
from glob import glob                                       # Busca de arquivos por padrões...
import seaborn as sns                                       # Visualização de dados estatísticos...
import pandas as pd                                         # Análise e manipulação de dados...
import numpy as np                                          # Cálculos numéricos e arrays...
import warnings                                             # Controle de avisos...
import base64                                               # Transformando o Relatório Específico em uma imagem...
import time                                                 # Visualizar se foi clicado no primeiro gráfico do Relatório Geral...
import io                                                   # Manipulação de streams em memória...
import os                                                   # Operações com arquivos e sistema...

pd.set_option('mode.chained_assignment', None)              # Desativa avisos de atribuição encadeada...
warnings.filterwarnings("ignore", category=FutureWarning)   # Oculta avisos de futura depreciação...

#### Consumindo os Dados dos arquivos Excel: Criado pelo Nicholas Birochi

In [ ]:
# Caminhos das pastas com os arquivos '.parquet':
pasta_executivos = R"..\..\..\Base de Dados Original\parquet\Executivos"
pasta_mensalistas = R"..\..\..\Base de Dados Original\parquet\Mensalistas"
pasta_horistas = R"..\..\..\Base de Dados Original\parquet\Horistas"
pasta_afastados = R"..\..\..\Base de Dados Original\parquet\Departamentos-Afastados\Afastados.parquet"
pasta_departamentos = R"..\..\..\Base de Dados Original\parquet\Departamentos-Afastados\Departamentos.parquet"
pasta_gptw = R"..\..\..\Base de Dados Original\parquet\GPTW\GPTW.parquet"

# Lista todos os arquivos '.parquet' em cada pasta:
arquivos_parquet_executivos = glob(os.path.join(pasta_executivos, "*.parquet"))
arquivos_parquet_mensalistas = glob(os.path.join(pasta_mensalistas, "*.parquet"))
arquivos_parquet_horistas = glob(os.path.join(pasta_horistas, "*.parquet"))

# Carrega os arquivos e adiciona a coluna 'Cargo':
tables = [
    pd.read_parquet(caminho).assign(Cargo=cargo)
    for arquivos, cargo in zip([arquivos_parquet_executivos, arquivos_parquet_mensalistas, arquivos_parquet_horistas], ['E', 'M', 'H'])
    for caminho in arquivos
]

# Carrega o arquivo para dentro do Data Frame:
afastados = pd.read_parquet(pasta_afastados)
departamentos = pd.read_parquet(pasta_departamentos)
gptw = pd.read_parquet(pasta_gptw)

#### Apresentando os valores da primeira tabela: Criado pelo Murilo

In [ ]:
# Primeira tabela:
tables[1]

## 2. Tratamento de Dados:

#### Definição das Funções para Transformação e Tratamento de Dados: Melhorado pelo Nicholas Birochi
1. **Renomear Colunas:**  
   Função que renomeia as colunas do DataFrame com base em um dicionário de mapeamento, garantindo padronização e clareza nos nomes.
2. **Mover Colunas:**  
   Função que move as colunas do DataFrame com base em um dicionário de mapeamento, garantindo padronização e clareza nos nomes.

In [ ]:
# Função para renomear coluna:
def renomear_coluna(df, col, newname):
    df = df.rename(columns={col: newname})
    return df

# Função para mover coluna
def mover_coluna(df, col, nova_posicao):
    colunas = list(df.columns)
    colunas.insert(nova_posicao, colunas.pop(colunas.index(col)))
    return df[colunas]

#### Aplicando Funções a lista de tabelas: Melhorado pelo Nicholas Birochi

In [ ]:
# Aplicando a função a todas as colunas de todas as tabelas:
for i in range(len(tables)):
    print(f"\rProcessando tabela {i+1}/{len(tables)}", end="", flush=True)
    tables[i] = renomear_coluna(tables[i], tables[i].columns[0], 'Cadastro')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[1], 'Coletor')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[2], 'Descricao Coletor')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[3], 'Nome')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[4], 'Filial')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[5], 'Data')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[6], 'Hora')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[7], 'Acesso')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[8], 'Dir')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[9], 'Tipo')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[10], 'Uso Acesso')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[11], 'Planta')
#'Cadastro', 'Nome', 'Filial', 'Data', 'Hora', 'Dir', 'Tipo', 'Acesso', 'Coletor', 'Planta', 'Descricao Coletor', 'Uso Acesso', 'Cargo'...

# Aplicando a função de renomear o nome da coluna nas colunas da tabela 'departamentos':
departamentos = renomear_coluna(departamentos, departamentos.columns[0], 'Cadastro')
departamentos = renomear_coluna(departamentos, departamentos.columns[2], 'Heranca')
departamentos = renomear_coluna(departamentos, departamentos.columns[3], 'Cargo Especifico')
departamentos = renomear_coluna(departamentos, departamentos.columns[4], 'Departamento')
departamentos = renomear_coluna(departamentos, departamentos.columns[5], 'Area')

# Aplicando a função de renomear o nome da coluna 'Departamento' da tabela 'afastados':
afastados = mover_coluna(afastados, 'Diretoria RH', 2)
afastados = renomear_coluna(afastados, afastados.columns[0], 'Cadastro')
afastados = renomear_coluna(afastados, afastados.columns[2], 'Heranca')
afastados = renomear_coluna(afastados, afastados.columns[3], 'Cargo Especifico')
afastados = renomear_coluna(afastados, afastados.columns[4], 'Departamento')
afastados = renomear_coluna(afastados, afastados.columns[5], 'Area')

# Aplicando a função de renomear o nome da coluna nas colunas da tabela 'gptw':
gptw = renomear_coluna(gptw, gptw.columns[0], 'Área')

# Visualizando a primeira tabela após o tratamento dela:
tables[0]

#### Concatenar todas as tabelas da lista em um único Data Frame: Melhorado pelo Nicholas Birochi

In [ ]:
# Concatenando todas as tabelas em um único DataFrame:
df_ronda = pd.concat(tables, ignore_index=True)

# Ordenando as colunas na ordem desejada:
df_ronda = df_ronda[['Cadastro', 'Nome', 'Filial', 'Data', 'Hora', 'Dir', 'Tipo', 'Acesso', 'Planta', 'Coletor', 'Descricao Coletor', 'Uso Acesso', 'Cargo']]
df_ronda = df_ronda.drop(df_ronda.index[-1]) # Remover última linha de total...

# Verificando o DataFrame final:
df_ronda.head()

#### Diferenciando todas as pessoas que foram demitidas ou que não serão analisadas: Criado pelo Nicholas Birochi

In [ ]:
afastados.head()

#### Inserindo todos os departamentos das pessoas que não foram demitidas: Criado pelo Nicholas Birochi

In [ ]:
departamentos.head()

In [ ]:
# 0) Normalização mínima do 'Cadastro' (evita 123, "123", 123.0 divergirem)
def _norm_id(s):
    s = s.astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    return s.replace({'nan': np.nan})  # volta NaN onde era NaN

for _df in (df_ronda, afastados, departamentos):
    if 'Cadastro' not in _df.columns:
        raise KeyError("Faltou a coluna 'Cadastro' em um dos DataFrames.")
    _df['Cadastro'] = _norm_id(_df['Cadastro'])

# 1) Afastado por Cadastro:
df_ronda['Afastado'] = df_ronda['Cadastro'].isin(afastados['Cadastro']).map({True: 'Sim', False: 'Não'})

# 2) Preparar 'departamentos' com nomes únicos de coluna:
deps = departamentos[['Cadastro','Nome completo','Area','Departamento','Heranca','Cargo Especifico']].rename(
    columns={
        'Nome completo':'Nome_dep',
        'Area':'Area_dep',
        'Departamento':'Departamento_dep',
        'Heranca':'Heranca_dep',
        'Cargo Especifico':'Cargo Especifico_dep'
    }
)

# 3) Merge (left) pelo Cadastro:
df_ronda = df_ronda.merge(deps, on='Cadastro', how='left')

# 4) Dicionários (afastados) por Cadastro:
map_afastados_nome = dict(zip(afastados['Cadastro'], afastados['Nome completo']))
map_afastados_area = dict(zip(afastados['Cadastro'], afastados['Area']))
map_afastados_dept = dict(zip(afastados['Cadastro'], afastados['Departamento']))
map_afastados_her  = dict(zip(afastados['Cadastro'], afastados['Heranca']))
map_afastados_carg = dict(zip(afastados['Cadastro'], afastados['Cargo Especifico']))

# 5) Substituir com prioridade: afastados > departamentos > original:

# Nome:
df_ronda['Nome'] = (
    df_ronda['Cadastro'].map(map_afastados_nome)   # 1º: afastados…
    .fillna(df_ronda['Nome_dep'])                  # 2º: departamentos…
    .fillna(df_ronda.get('Nome'))                  # 3º: valor original (se existir)…
)

# Area:
df_ronda['Area'] = (
    df_ronda['Cadastro'].map(map_afastados_area)
    .fillna(df_ronda['Area_dep'])
)

# Departamento:
df_ronda['Departamento'] = (
    df_ronda['Cadastro'].map(map_afastados_dept)
    .fillna(df_ronda['Departamento_dep'])
)

# Heranca:
df_ronda['Heranca'] = (
    df_ronda['Cadastro'].map(map_afastados_her)
    .fillna(df_ronda['Heranca_dep'])
)

# Cargo Especifico:
df_ronda['Cargo Especifico'] = (
    df_ronda['Cadastro'].map(map_afastados_carg)
    .fillna(df_ronda['Cargo Especifico_dep'])
)

# 6) (Opcional) Manter as colunas *_dep para auditoria; se quiser remover, descomente abaixo:
df_ronda = df_ronda.drop(columns=['Nome_dep','Area_dep','Departamento_dep','Heranca_dep','Cargo Especifico_dep'], errors='ignore')

# 7) Visualizando o resultado final:
df_ronda

## 3. Analisando os Dados:

#### Formatando tipos de dados: Criado pelo Murilo

In [ ]:
# Visualizando os tipos de dados:
df_ronda.dtypes

#### Descrobrindo todos os valores distintos possíveis de Hora: Melhorado pelo Nicholas Birochi

In [ ]:
# Detecta se a coluna contém inteiros (minutos desde 00:00):
if pd.api.types.is_integer_dtype(df_ronda['Hora']) or df_ronda['Hora'].astype(str).str.fullmatch(r'\d+').all():
    # Converte os minutos inteiros para timedelta:
    df_ronda['Hora'] = pd.to_timedelta(df_ronda['Hora'], unit='m')

# Garante que seja timedelta:
df_ronda['Hora'] = pd.to_timedelta(df_ronda['Hora'])

# Remove tudo que não é 'hora', 'minuto', 'segundos' e 'milissegundos':
df_ronda['Hora'] = df_ronda['Hora'].dt.components.apply(
    lambda row: f"{int(row.hours):02d}:{int(row.minutes):02d}", axis=1
)

#### Descrobrindo todos os valores distintos possíveis de Data: Melhorado pelo Nicholas Birochi

In [ ]:
# Garante que seja datetime:
df_ronda['Data'] = pd.to_datetime(df_ronda['Data'])

# Formata para 'dd/mm/yyyy':
df_ronda['Data'] = df_ronda['Data'].dt.strftime('%d/%m/%Y')

# Garantir datetime nos dados de admissão/demissão:
afastados['Data_Adm'] = pd.to_datetime(afastados['Data_Adm'])
afastados['Data_Dem'] = pd.to_datetime(afastados['Data_Dem'])
departamentos['Data_Admissão'] = pd.to_datetime(departamentos['Data_Admissão'], format='%d.%m.%Y')

#### Corrigindo o tipo de dado de cada coluna do Data Frame: Melhorado pelo Nicholas Birochi

In [ ]:
# Formatar categóricos numéricos, como interger, para ficar mais bonito visualmente e formatando todas as outras colunas de dentro de 'df_ronda':
df_ronda['Cadastro'] = df_ronda['Cadastro'].astype(str)
df_ronda['Nome'] = df_ronda['Nome'].astype(str)
df_ronda['Filial'] = df_ronda['Filial'].astype(str)
df_ronda['Dir'] = df_ronda['Dir'].astype(str)
df_ronda['Tipo'] = df_ronda['Tipo'].astype(int)
df_ronda['Acesso'] = df_ronda['Acesso'].astype(str)
df_ronda['Planta'] = df_ronda['Planta'].astype(str)
df_ronda['Coletor'] = df_ronda['Coletor'].astype(str)
df_ronda['Descricao Coletor'] = df_ronda['Descricao Coletor'].astype(str)
df_ronda['Uso Acesso'] = df_ronda['Uso Acesso'].astype(int)
df_ronda['Cargo'] = df_ronda['Cargo'].astype(str)
df_ronda['Afastado'] = df_ronda['Afastado'].astype(str)
df_ronda['Area'] = df_ronda['Area'].astype(str)
df_ronda['Departamento'] = df_ronda['Departamento'].astype(str)
df_ronda['Heranca'] = df_ronda['Heranca'].astype(str)
df_ronda['Cargo Especifico'] = df_ronda['Cargo Especifico'].astype(str)

# Corrigindo os tipos das colunas de 'departamentos':
departamentos['Cadastro'] = departamentos['Cadastro'].astype(str)

# Demonstrando os tipos de dados que são de cada coluna:
df_ronda.dtypes

#### Visualizando os nomes das colunas: Criado pelo Murilo

In [ ]:
# Visualizando os nomes das colunas do Data Frame:
df_ronda.head()

#### Adicionando a coluna 'Dia Semana': Criado pelo Murilo

In [ ]:
# Inserindo os dias da semana, assim garante que a coluna 'Data' seja datetime:
df_ronda['Data'] = pd.to_datetime(df_ronda['Data'], dayfirst=True)

# Cria a coluna com o nome do dia da semana (em inglês):
df_ronda['Dia Semana'] = df_ronda['Data'].dt.strftime('%A')

# Visualizando o resultado:
df_ronda.head()

#### Alterando a ordem das colunas: Melhorado pelo Nicholas Birochi

In [ ]:
# Definir base para realização dos cálculos de presença na fábrica:
df_ronda = df_ronda[['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Dir', 'Tipo', 'Acesso', 'Planta', 'Coletor', 'Descricao Coletor', 'Uso Acesso', 'Cargo', 'Afastado', 'Dia Semana', 'Data', 'Hora']]

# Visualizando o resultado:
df_ronda

# Integração Analítica Final:
#### Realização da carga final dos dados pós-tratamento na camada analítica (OLAP), visando disponibilização para consultas multidimensionais:

#### Fazendo o Data Frame para o presencial: Melhorado pelo Nicholas Birochi

In [ ]:
# Determinar df de entrada presencial - Planta != '999':
df_presencial = df_ronda[(df_ronda['Planta'] != '999')]

# Filtrando o primeiro e o último horário de cada dia:
primeiros_horarios = df_presencial.groupby(['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Afastado', 'Data'])['Hora'].min().reset_index()
ultimos_horarios = df_presencial.groupby(['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Afastado', 'Data'])['Hora'].max().reset_index()

# Mesclando os primeiros e últimos horários de cada dia:
df_presencial_filtrado = pd.merge(primeiros_horarios, ultimos_horarios, on=['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Afastado', 'Data'], suffixes=('_primeiro', '_ultimo'))

# Mesclando os registros originais com os primeiros e últimos registros de ponto:
df_presencial_filtrado = pd.merge(df_presencial, df_presencial_filtrado, on=['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Afastado', 'Data'])

# Mantendo somente os registros que correspondem ao primeiro ou último horário de cada dia:
df_presencial_filtrado = df_presencial_filtrado[(df_presencial_filtrado['Hora'] == df_presencial_filtrado['Hora_primeiro']) | (df_presencial_filtrado['Hora'] == df_presencial_filtrado['Hora_ultimo'])]

# Removendo colunas desnecessárias:
df_presencial_filtrado = df_presencial_filtrado.drop(columns=['Hora_primeiro', 'Hora_ultimo'])

# Tabela Final:
df_presencial_filtrado.head()

#### Fazendo o Data Frame para o home office: Criado pelo Nicholas Birochi

In [ ]:
'''# Determinar df de entrada home office - Planta == '999':
df_homeoffice = df_ronda[(df_ronda['Planta'] == '999')]

# Filtrando o primeiro e o último horário de cada dia:
primeiros_horarios = df_homeoffice.groupby(['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Afastado', 'Data'])['Hora'].min().reset_index()
ultimos_horarios = df_homeoffice.groupby(['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Afastado', 'Data'])['Hora'].max().reset_index()

# Mesclando os primeiros e últimos horários de cada dia:
df_homeoffice_filtrado = pd.merge(primeiros_horarios, ultimos_horarios, on=['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Afastado', 'Data'], suffixes=('_primeiro', '_ultimo'))

# Mesclando os registros originais com os primeiros e últimos registros de ponto:
df_homeoffice_filtrado = pd.merge(df_homeoffice, df_homeoffice_filtrado, on=['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Afastado', 'Data'])

# Mantendo somente os registros que correspondem ao primeiro ou último horário de cada dia:
df_homeoffice_filtrado = df_homeoffice_filtrado[(df_homeoffice_filtrado['Hora'] == df_homeoffice_filtrado['Hora_primeiro']) | (df_homeoffice_filtrado['Hora'] == df_homeoffice_filtrado['Hora_ultimo'])]

# Removendo colunas desnecessárias:
df_homeoffice_filtrado = df_homeoffice_filtrado.drop(columns=['Hora_primeiro', 'Hora_ultimo'])

# Tabela Final:
df_homeoffice_filtrado'''

#### Finalizando os tratamentos, mas já dentro do Data Frame final: Melhorado pelo Nicholas Birochi

In [ ]:
# Criar um df com 'Data Real' distinta de cada pessoa - removendo duplicatas para obter a quantidade distinta de dias de presença na planta de cada pessoa somente para o presencial:
df_datas = df_presencial_filtrado.drop_duplicates(subset = ['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Afastado', 'Data'])

# Visualizando os dados:
df_datas.head()

#### Finalizando os tratamentos, mas já dentro do Data Frame final para o Home Office: Criado pelo Nicholas Birochi

In [ ]:
'''# Criar um df com 'Data Real' distinta de cada pessoa - removendo duplicatas para obter a quantidade distinta de dias de presença na planta de cada pessoa somente para o Home Office:
df_homeoffice_datas = df_homeoffice_filtrado.drop_duplicates(subset = ['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Afastado', 'Data'])

# Visualizando os dados:
df_homeoffice_datas'''

#### Fazendo uma análise da presença semanal de cada pessoa: Melhorado pelo Nicholas Birochi

In [ ]:
# Fazer um count de 'Dia Semana' para compreender maior porcentagem de presença por dias:
df_count_por_semana = df_datas.groupby(['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Afastado'])['Dia Semana'].value_counts().reset_index()
df_count_por_semana.columns = ['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Afastado', 'Dia Semana', 'Contagem Presença']
df_count_por_semana.head()

## Criando o primeiro gráfico para o presencial (Mapa de Calor): Criado pelo Nicholas Birochi

In [ ]:
# Contar total de presenças por pessoa (não repetir):
contagem_total_por_pessoa = (
    df_count_por_semana.groupby('Cadastro')['Contagem Presença']
    .sum()  # Soma dos valores da coluna 'Contagem Presença'...
    .reset_index(name='Total de Presenças')
)

# Juntar com o DataFrame de nomes, garantindo que não haja duplicação de pessoas:
contagem_total_por_pessoa = contagem_total_por_pessoa.merge(
    df_ronda,
    on='Cadastro',
    how='left'
)

# Criar coluna com Cadastro + Nome:
contagem_total_por_pessoa['Cadastro_Nome'] = contagem_total_por_pessoa['Cadastro'].astype(str) + ' - ' + contagem_total_por_pessoa['Nome']

# Ordenar pelo total crescente:
contagem_total_por_pessoa = contagem_total_por_pessoa.sort_values('Total de Presenças')

# Selecionar as 100 pessoas com menor presença (sem repetições):
contagem_total_por_pessoa = contagem_total_por_pessoa.drop_duplicates(subset='Cadastro').head(100)

# Criar DataFrame para heatmap:
heatmap_data_pivot = contagem_total_por_pessoa.set_index('Cadastro_Nome')[['Total de Presenças']].rename(columns={'Total de Presenças': 'Total'})

# Obter o valor máximo baseado na soma total das presenças por pessoa:
valor_maximo_soma_presencas = contagem_total_por_pessoa['Total de Presenças'].max()

# Tamanho proporcional:
altura = min(max(heatmap_data_pivot.shape[0] * 0.3, 6), 50)

# Criar figura:
plt.figure(figsize=(4, altura))
sns.heatmap(
    heatmap_data_pivot,
    annot=True,
    fmt='g',
    cmap='YlOrRd_r',
    linewidths=0.4,
    linecolor='white',
    cbar_kws={'label': 'Contagem de Presença'},
    annot_kws={'fontsize': 8, 'color': 'black'},
    vmin=0,                           # Mínimo sempre '0'...
    vmax=valor_maximo_soma_presencas  # Valor máximo baseado na soma total das presenças por pessoa...
)

plt.title('100 Pessoas com Menor Presença Total no Presencial', fontsize=14, fontweight='bold')
plt.xlabel('', fontsize=12)
plt.ylabel('Pessoa (Cadastro - Nome)', fontsize=12)
plt.xticks(rotation=0)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

#### Para Home Office: Criado pelo Nicholas Birochi

In [ ]:
'''# Fazer um count de 'Dia_Semana' para compreender maior porcentagem de presença por dias:
df_homeoffice_count = df_homeoffice_datas.groupby('Cadastro')['Dia Semana'].value_counts().reset_index()
df_homeoffice_count.columns = ['Cadastro', 'Dia Semana', 'Contagem Presença']
df_homeoffice_count'''

#### Inserindo a coluna para visualizar as datas de maneira mais dinâmica: Criado pelo Murilo

In [ ]:
# Garante que a coluna 'Data' esteja em formato datetime:
df_datas['Data'] = pd.to_datetime(df_datas['Data'], dayfirst=True)
#df_homeoffice_datas['Data'] = pd.to_datetime(df_homeoffice_datas['Data'], dayfirst=True)

# Agora pode usar .dt normalmente:
df_datas['Semana por Ano'] = df_datas['Data'].dt.isocalendar().week
#df_homeoffice_datas['Semana por Ano'] = df_homeoffice_datas['Data'].dt.isocalendar().week

# Visualizando o Data Frame já preenchido:
df_datas.head()

#### Análise dos dias presenciais por semana: Criado pelo Nicholas Birochi

In [ ]:
# Criando a coluna 'Ano':
df_datas['Ano'] = df_datas['Data'].dt.year
#df_homeoffice_datas['Ano'] = df_homeoffice_datas['Data'].dt.year

# Definir base para realização dos cálculos de presença na fábrica:
df_datas = df_datas[['Nome', 'Cadastro', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Dir', 'Tipo', 'Acesso', 'Planta', 'Coletor', 'Descricao Coletor', 'Uso Acesso', 'Cargo', 'Afastado', 'Hora', 'Data', 'Dia Semana', 'Semana por Ano', 'Ano']]
#df_homeoffice_datas = df_homeoffice_datas[['Nome', 'Cadastro', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Dir', 'Tipo', 'Acesso', 'Planta', 'Coletor', 'Descricao Coletor', 'Uso Acesso', 'Cargo', 'Afastado', 'Hora', 'Data', 'Dia Semana', 'Semana por Ano', 'Ano']]

# Visualizando o Data Frame já ordenado:
df_datas.head()

#### Fazendo a contagem total da presença por dias: Melhorado pelo Nicholas Birochi

In [ ]:
# Fazer um count de 'Semana Ano' para compreender maior porcentagem de presença por dias:
df_count_semana = df_datas[['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Cargo', 'Afastado', 'Ano', 'Semana por Ano']].value_counts().reset_index()
df_count_semana.columns = ['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Cargo', 'Afastado', 'Ano', 'Semana por Ano', 'Contagem Presença']
df_count_semana = df_count_semana.sort_values(by=['Ano', 'Semana por Ano']).reset_index(drop=True)

# Visualizando o Data Frame preenchido com a quantidade de dias de presença por semana para cada pessoa, no presencial:
df_count_semana.head()

#### Fazendo a contagem total da presença por dias: Criado pelo Nicholas Birochi

In [ ]:
'''# Fazer um count de 'Semana por Ano' para compreender maior porcentagem de Home Office por dias:
df_homeoffice_count_semana = df_homeoffice_datas[['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Cargo', 'Afastado', 'Ano', 'Semana por Ano']].value_counts().reset_index()
df_homeoffice_count_semana.columns = ['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Cargo', 'Afastado', 'Ano', 'Semana por Ano', 'Contagem Home Office']
df_homeoffice_count_semana = df_homeoffice_count_semana.sort_values(by=['Ano', 'Semana por Ano']).reset_index(drop=True)

# Visualizando o Data Frame preenchido com a quantidade de dias de presença por semana para cada pessoa, no presencial:
df_homeoffice_count_semana'''

#### Cria uma tabela somente para o filtro das datas: Criado pelo Murilo

In [ ]:
# Calculando os dais úteis entre o período de Janeiro de 2024 até Março de 2025:
# Datas de início e fim do período:
inicio = '2024-01-01'
fim = '2025-03-31'

# Gera a lista de datas com frequência semanal (toda segunda-feira):
datas_semanais = pd.date_range(start = inicio, end = fim, freq = 'W-MON')

# Cria a tabela com início e fim de cada semana:
tabela_semanas = pd.DataFrame({
    'Início da Semana': datas_semanais,
    'Fim da Semana': datas_semanais + timedelta(days=6)
})

# Adiciona a coluna com o número da semana (ISO):
tabela_semanas['Ano'] = tabela_semanas['Início da Semana'].dt.isocalendar().year
tabela_semanas['Semana por Ano'] = tabela_semanas['Início da Semana'].dt.isocalendar().week

# Exibe a tabela:
tabela_semanas

#### Continua modulando o filtro por datas para o presencial: Criado pelo Murilo

In [ ]:
# Faz o merge com base em 'Cadastro', 'Ano' e 'Número da Semana':
df_semana_merged = pd.merge(
    tabela_semanas,
    df_count_semana,
    on=['Ano', 'Semana por Ano'],
    how='left'  # Mantém todas as semanas da tabela_semanas...
)

# Preenche valores ausentes com zero:
df_semana_merged['Contagem Presença'] = df_semana_merged['Contagem Presença'].fillna(0).astype(int)
df_semana_merged = df_semana_merged.sort_values(by='Contagem Presença', ascending=False)

# Exibe a tabela final:
df_semana_merged.head()

#### Definir períodos de férias a serem removidos: Melhorado pelo Nicholas Birochi

In [ ]:
# Datas das férias coletivas:
ferias = [
    (datetime(2023, 12, 18), datetime(2024, 1, 7)),
    (datetime(2024, 12, 23), datetime(2025, 1, 12)),
]

# Função para checar se uma semana está em férias:
def esta_em_ferias(inicio_semana):
    for inicio, fim in ferias:
        if inicio <= inicio_semana <= fim:
            return True
    return False

# Garantir que as datas sejam datetime:
df_semana_merged['Início da Semana'] = pd.to_datetime(df_semana_merged['Início da Semana'])
df_semana_merged['Fim da Semana'] = pd.to_datetime(df_semana_merged['Fim da Semana'])

# Filtra fora das férias:
df_filtrado = df_semana_merged[
    ~df_semana_merged['Início da Semana'].apply(esta_em_ferias)
]

# Criar lista de todas as segundas-feiras entre min e max:
data_min = df_filtrado['Início da Semana'].min()
data_max = df_filtrado['Início da Semana'].max()
todas_semanas = pd.date_range(start=data_min, end=data_max, freq='W-MON')

# Remover semanas em férias e semana 1 (se desejar):
todas_semanas = [
    df for df in todas_semanas
    if not esta_em_ferias(df) and df.isocalendar()[1] != 1
]

# Criar DataFrame de semanas com Ano e Semana por Ano:
semanas_df = pd.DataFrame({
    'Início da Semana': todas_semanas
})
semanas_df['Fim da Semana'] = semanas_df['Início da Semana'] + pd.to_timedelta(6, unit='d')
semanas_df['Ano'] = semanas_df['Início da Semana'].dt.isocalendar().year
semanas_df['Semana por Ano'] = semanas_df['Início da Semana'].dt.isocalendar().week

# Lista única de pessoas e nomes:
pessoas_df = df_filtrado[['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Cargo', 'Afastado']].drop_duplicates()

# Produto cartesiano: todas pessoas × todas semanas:
combinacoes = pd.merge(
    pessoas_df.assign(key=1),
    semanas_df.assign(key=1),
    on='key'
).drop('key', axis=1)

# Merge com dados reais:
df_semana_merged = pd.merge(
    combinacoes,
    df_filtrado[['Cadastro', 'Início da Semana', 'Contagem Presença']],
    on=['Cadastro', 'Início da Semana'],
    how='left'
)

# Preencher ausências com zero:
df_semana_merged['Contagem Presença'] = df_semana_merged['Contagem Presença'].fillna(0).astype(int)

# Ordenar cronologicamente:
df_semana_merged = df_semana_merged.sort_values(['Cadastro', 'Início da Semana']).reset_index(drop=True)

# (opcional) remover duplicados para evitar erro no map:
afastados_dedup = afastados.drop_duplicates(subset='Cadastro', keep='first')
departamentos_dedup = departamentos.drop_duplicates(subset='Cadastro', keep='first')

# Remover registros da última semana:
ultima_semana = semanas_df['Início da Semana'].max()
df_semana_merged = df_semana_merged[df_semana_merged['Início da Semana'] < ultima_semana]

# Mapear datas de afastados e departamentos com índice único
df_semana_merged['Data_Adm'] = df_semana_merged['Cadastro'].map(
    afastados_dedup.set_index('Cadastro')['Data_Adm']
)
df_semana_merged['Data_Dem'] = df_semana_merged['Cadastro'].map(
    afastados_dedup.set_index('Cadastro')['Data_Dem']
)
df_semana_merged['Data_Admissão'] = df_semana_merged['Cadastro'].map(
    departamentos_dedup.set_index('Cadastro')['Data_Admissão']
)

# Filtrar registros dentro do período válido:
df_semana_merged = df_semana_merged[
    (
        df_semana_merged['Data_Adm'].notna() &
        (df_semana_merged['Início da Semana'] >= df_semana_merged['Data_Adm']) &
        (
            df_semana_merged['Data_Dem'].isna() |
            (df_semana_merged['Início da Semana'] <= df_semana_merged['Data_Dem'])
        )
    )
    |
    (
        df_semana_merged['Data_Adm'].isna() &
        df_semana_merged['Data_Admissão'].notna() &
        (df_semana_merged['Início da Semana'] >= df_semana_merged['Data_Admissão'])
    )
]

# Remover colunas temporárias:
df_semana_merged = df_semana_merged.drop(columns=['Data_Adm', 'Data_Dem', 'Data_Admissão'])

# Exibir resultado:
df_semana_merged.head()

#### Semanas com somente 'um' ou 'zero' de registro de dia presencial: Criado pelo Murilo

In [ ]:
# Filtra o DataFrame para manter apenas os registros com 'Contagem Presença' igual a 0 ou 1, correspondendo a pessoas com no máximo um registro de presença no dia e reseta o índice para manter a consistência após o filtro:
df_semana_merged_dia = df_semana_merged[(df_semana_merged['Contagem Presença'] == 1) | (df_semana_merged['Contagem Presença'] == 0)].reset_index(drop=True)

# Remove semana 1 por ser coletivas:
df_semana_merged_dia = df_semana_merged_dia[df_semana_merged_dia['Semana por Ano'] != 1].reset_index(drop=True)
df_semana_merged_dia.head()

#### Continua modulando o filtro por datas para o home office: Criado pelo Nicholas Birochi

In [ ]:
'''# Faz o merge com base em 'Cadastro', 'Ano' e 'Número da Semana':
df_homeoffice_semana_merged = pd.merge(
    tabela_semanas,
    df_homeoffice_count_semana,
    on=['Ano', 'Semana por Ano'],
    how='left'  # Mantém todas as semanas da tabela_semanas...
)

# Preenche valores ausentes com zero:
df_homeoffice_semana_merged['Contagem Home Office'] = df_homeoffice_semana_merged['Contagem Home Office'].fillna(0).astype(int)
df_homeoffice_semana_merged = df_homeoffice_semana_merged.sort_values(by='Contagem Home Office', ascending=False)

# Exibe a tabela final:
df_homeoffice_semana_merged'''

#### Corrigindo o formato certo para a coluna: Criado pelo Nicholas Birochi

In [ ]:
# Normaliza tabela_semanas:
tabela_semanas["Ano"] = tabela_semanas["Ano"].astype("int32")
tabela_semanas["Semana por Ano"] = tabela_semanas["Semana por Ano"].astype("int32")

'''# Normaliza df_homeoffice_count_semana:
df_homeoffice_count_semana["Ano"] = df_homeoffice_count_semana["Ano"].astype("int32")
df_homeoffice_count_semana["Semana por Ano"] = df_homeoffice_count_semana["Semana por Ano"].astype("int32")'''

#### Tratando o período útil a ser utilizado: Criado pelo Nicholas Birochi

In [ ]:
'''# Datas das férias coletivas:
ferias = [
    (datetime(2023, 12, 18), datetime(2024, 1, 7)),
    (datetime(2024, 12, 23), datetime(2025, 1, 12)),
]

# Função para checar se uma semana está em férias:
def esta_em_ferias(inicio_semana):
    for inicio, fim in ferias:
        if inicio <= inicio_semana <= fim:
            return True
    return False

# Garantir que as datas sejam datetime:
df_homeoffice_semana_merged['Início da Semana'] = pd.to_datetime(df_homeoffice_semana_merged['Início da Semana'])
df_homeoffice_semana_merged['Fim da Semana'] = pd.to_datetime(df_homeoffice_semana_merged['Fim da Semana'])

# Filtra fora das férias:
df_homeoffice_semana_merged = df_homeoffice_semana_merged[
    ~df_homeoffice_semana_merged['Início da Semana'].apply(esta_em_ferias)
]

# Criar lista de todas as segundas-feiras entre min e max:
data_min = df_homeoffice_semana_merged['Início da Semana'].min()
data_max = df_homeoffice_semana_merged['Início da Semana'].max()

todas_semanas = pd.date_range(start=data_min, end=data_max, freq='W-MON')

# Remover semanas em férias e semana 1 (se desejar):
todas_semanas = [
    d for d in todas_semanas
    if not esta_em_ferias(d) and d.isocalendar()[1] != 1
]

# Criar DataFrame de semanas com Ano e Semana por Ano:
semanas_df = pd.DataFrame({
    'Início da Semana': todas_semanas
})
semanas_df['Fim da Semana'] = semanas_df['Início da Semana'] + pd.to_timedelta(6, unit='d')
semanas_df['Ano'] = semanas_df['Início da Semana'].dt.isocalendar().year
semanas_df['Semana por Ano'] = semanas_df['Início da Semana'].dt.isocalendar().week

# Lista única de pessoas e nomes:
pessoas_df = df_homeoffice_semana_merged[['Cadastro', 'Nome', 'Cargo']].drop_duplicates()

# Produto cartesiano: todas pessoas × todas semanas:
combinacoes = pd.merge(
    pessoas_df.assign(key=1),
    semanas_df.assign(key=1),
    on='key'
).drop('key', axis=1)

# Merge com dados reais:
df_homeoffice_semana_merged = pd.merge(
    combinacoes,
    df_homeoffice_semana_merged[['Cadastro', 'Início da Semana', 'Contagem Home Office']],
    on=['Cadastro', 'Início da Semana'],
    how='left'
)

# Preencher ausências com zero:
df_homeoffice_semana_merged['Contagem Home Office'] = df_homeoffice_semana_merged['Contagem Home Office'].fillna(0).astype(int)

# Ordenar cronologicamente:
df_homeoffice_semana_merged = df_homeoffice_semana_merged.sort_values(['Cadastro', 'Início da Semana']).reset_index(drop=True)

# Identifica a última data de início da semana:
ultima_semana = df_homeoffice_semana_merged['Início da Semana'].max()

# Remove registros da última semana:
df_homeoffice_semana_merged = df_homeoffice_semana_merged[df_homeoffice_semana_merged['Início da Semana'] != ultima_semana]

# (opcional) remover duplicados para evitar erro no map:
afastados_dedup = afastados.drop_duplicates(subset='Cadastro', keep='first')
departamentos_dedup = departamentos.drop_duplicates(subset='Cadastro', keep='first')

# Mapear datas de afastados e departamentos com índice único
df_homeoffice_semana_merged['Data_Adm'] = df_homeoffice_semana_merged['Cadastro'].map(
    afastados_dedup.set_index('Cadastro')['Data_Adm']
)
df_homeoffice_semana_merged['Data_Dem'] = df_homeoffice_semana_merged['Cadastro'].map(
    afastados_dedup.set_index('Cadastro')['Data_Dem']
)
df_homeoffice_semana_merged['Data_Admissão'] = df_homeoffice_semana_merged['Cadastro'].map(
    departamentos_dedup.set_index('Cadastro')['Data_Admissão']
)

# Filtrar registros dentro do período válido
df_homeoffice_semana_merged = df_homeoffice_semana_merged[
    (
        df_homeoffice_semana_merged['Data_Adm'].notna() &
        (df_homeoffice_semana_merged['Início da Semana'] >= df_homeoffice_semana_merged['Data_Adm']) &
        (
            df_homeoffice_semana_merged['Data_Dem'].isna() |
            (df_homeoffice_semana_merged['Início da Semana'] <= df_homeoffice_semana_merged['Data_Dem'])
        )
    )
    |
    (
        df_homeoffice_semana_merged['Data_Adm'].isna() &
        df_homeoffice_semana_merged['Data_Admissão'].notna() &
        (df_homeoffice_semana_merged['Início da Semana'] >= df_homeoffice_semana_merged['Data_Admissão'])
    )
]

# Remover colunas temporárias
df_homeoffice_semana_merged = df_homeoffice_semana_merged.drop(columns=['Data_Adm', 'Data_Dem', 'Data_Admissão'])

# Exibir resultado:
df_homeoffice_semana_merged'''

#### Semanas com somente 'um' ou 'zero' de registro de dia home office: Criado pelo Nicholas Birochi

In [ ]:
# Filtra o DataFrame para manter apenas os registros com 'Contagem Presença' igual a 0 ou 1, correspondendo a pessoas com no máximo um registro de presença no dia e reseta o índice para manter a consistência após o filtro:
df_semana_merged_dia = df_semana_merged[(df_semana_merged['Contagem Presença'] == 1) | (df_semana_merged['Contagem Presença'] == 0)].reset_index(drop=True)

# Remove semana 1 por ser coletivas:
df_semana_merged_dia = df_semana_merged_dia[df_semana_merged_dia['Semana por Ano'] != 1].reset_index(drop=True)
df_semana_merged_dia.head()

# Requisição por Chapa: Criado pelo Nicholas Birochi

In [ ]:
# Fazer um count de 'Semana Ano' para compreender maior porcentagem de presença por dias:
df_count_semana = df_datas[['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Cargo', 'Afastado', 'Semana por Ano', 'Ano']].value_counts().reset_index()
df_count_semana.columns = ['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Cargo Especifico', 'Cargo', 'Afastado', 'Semana por Ano', 'Ano', 'Contagem Presença']
df_count_semana = df_count_semana.sort_values(by=['Ano', 'Semana por Ano']).reset_index(drop=True)
df_count_semana.head()

#### Exibindo a presença por pessoa no horário de serviço: Criado pelo Nicholas Birochi

In [ ]:
# Merge com base em 'Ano', 'Semana por Ano', 'Cadastro' e 'Nome':
df_presenca_merged_por_cadastro = df_semana_merged

# Preenche valores ausentes com zero:
df_presenca_merged_por_cadastro['Contagem Presença'] = df_presenca_merged_por_cadastro['Contagem Presença'].fillna(0).astype(int)

# Agrupa por pessoa (Cadastro) somando as presenças:
df_presenca_merged = df_presenca_merged_por_cadastro.groupby(['Cadastro', 'Nome', 'Cargo', 'Afastado'])['Contagem Presença'].sum().reset_index()

# Ordena do maior para o menor:
df_presenca_merged = df_presenca_merged.sort_values(by = 'Contagem Presença', ascending = False)

# Exibe o resultado:
df_presenca_merged.head()

## 2. Criando o Gráfico Geral (Gráfico de Pizza, Gráfico de Linha e Relação entre Mensalistas e Executivos): Criado pelo Nicholas Birochi

In [ ]:
# Constantes globais
estilo_oculto = {'display': 'none'}
estilo_exibir = {'display': 'block', 'padding': '20px', 'backgroundColor': 'white'}
ordenacao_dias = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado', 'Domingo']
dias_portugues = {'Monday': 'Segunda', 'Tuesday': 'Terça', 'Wednesday': 'Quarta', 'Thursday': 'Quinta', 'Friday': 'Sexta', 'Saturday': 'Sábado', 'Sunday': 'Domingo'}

her = (
    df_semana_merged[['Area', 'Departamento', 'Heranca']]
    .drop_duplicates()
    .sort_values('Area')
)

dropdown_options = [{'label': 'Geral (todas as áreas)', 'value': 'GERAL'}] + [
    {'label': f"{row['Area']} {row['Departamento']}", 'value': row['Area']}
    for _, row in her.iterrows()
]

# Helpers / constantes para o RangeSlider de meses
LT_INICIO_ABS = dt.date(2024, 1, 1)      # início absoluto da régua
LT_FIM_ABS = dt.date.today()             # teto dinâmico = hoje

# jan/2024 .. hoje (primeiro dia de cada mês)
LT_MONTHS = pd.date_range(LT_INICIO_ABS, LT_FIM_ABS, freq="MS").date

def _lt_idx_of(y, m):
    return next(i for i, d in enumerate(LT_MONTHS) if d.year == y and d.month == m)

# Cenários
LT_C1_MIN = _lt_idx_of(2024, 1)  # jan/2024
LT_C1_MAX = _lt_idx_of(2025, 3)  # mar/2025
LT_C2_MIN = _lt_idx_of(2025, 4)  # abr/2025
LT_C2_MAX = len(LT_MONTHS) - 1   # último mês gerado

# Marks trimestrais p/ não poluir
def _lt_label(d): return d.strftime("%b/%Y").lower()
LT_MARKS = {i: (_lt_label(d) if d.month in [1,4,7,10] else "") for i, d in enumerate(LT_MONTHS)}

app = Dash(__name__, suppress_callback_exceptions=True)

app.layout =html.Div([
    dcc.Location(id='url', refresh=False),
    html.Div(
        # --- Página Inicial ---
        dcc.Loading(
            id="loading-geral",
            type="circle",
            fullscreen=True,
            children=html.Div([
                html.Div(id='pagina-geral', children=[
                    html.Div([
                        html.H1(["Relatório Presencial Geral de toda a Planta (Mensalistas/Executivos)",
                                html.Br(),
                                html.Span(id='periodo-texto')
                        ],style={'textAlign': 'center', 'flex': '1'})
                    ], style={'display': 'flex', 'justifyContent': 'flex-end', 'alignItems': 'center', 'marginBottom': '20px'}),
                    # ================== BLOCO DE FILTROS ==================
                    html.Div([
                        html.Div([
                            # --- Bloco A: 2 RadioItems ---
                            html.Div([
                                html.Label("Mostrar as pessoas afastadas?"),
                                dcc.RadioItems(
                                    id='afastados-toggle',
                                    options=[
                                        {'label': 'Sim', 'value': 'left'},
                                        {'label': 'Não', 'value': 'right'}
                                    ],
                                    value='left',
                                    labelStyle={'display': 'inline-block', 'marginRight': '10px'}
                                ),
                            ], style={'display': 'flex', 'flexDirection': 'column', 'alignItems': 'center'}),

                            html.Div([
                                html.Label("Mostrar as áreas que viajam?"),
                                dcc.RadioItems(
                                    id='viajam-toggle',
                                    options=[
                                        {'label': 'Sim', 'value': 'left'},
                                        {'label': 'Não', 'value': 'right'}
                                    ],
                                    value='left',
                                    labelStyle={'display': 'inline-block', 'marginRight': '10px'}
                                ),
                            ], style={'display': 'flex', 'flexDirection': 'column', 'alignItems': 'center'}),
                        ], style={'display': 'flex', 'gap': '30px', 'alignItems': 'center'}),

                        # --- Bloco B: Linha do tempo ---
                        html.Div([
                            html.Div([
                                dcc.RadioItems(
                                    id='cenario-toggle',
                                    options=[
                                        {'label': 'Jan/2024 a Mar/2025', 'value': 'c1'},
                                        {'label': 'Abr/2025 em diante', 'value': 'c2'}
                                    ],
                                    value='c1',
                                    labelStyle={'display': 'inline-block'}
                                )
                            ], style={'textAlign': 'center', 'marginBottom': '6px'}),

                            dcc.RangeSlider(
                                id='periodo-slider',
                                min=0,
                                max=len(LT_MONTHS)-1,
                                value=[LT_C1_MIN, LT_C1_MAX],
                                marks=LT_MARKS,
                                allowCross=False,
                                step=1,
                                pushable=1
                            )
                        ], style={'marginLeft': '50px', 'flex': '0 0 560px'}),

                        # --- Bloco C: Dropdown ---
                        html.Div([
                            dcc.Dropdown(
                                id='dropdown-area',
                                options=dropdown_options,
                                value='GERAL',
                                clearable=False,
                                style={'width': '300px'}
                            ),
                            html.Div([
                                dcc.Dropdown(
                                    id='dropdown-subarea',
                                    options=[],
                                    value=None,
                                    placeholder='Selecione uma subárea',
                                    clearable=True,
                                    style={'width': '300px', 'marginTop': '10px', 'display': 'none'}
                                )
                            ], id='subarea-container')
                        ], style={'marginLeft': 'auto'})
                    ], style={
                        'flex': '1',
                        'padding': '20px',
                        'boxSizing': 'border-box',
                        'backgroundColor': "#e0e0e0",
                        'borderRadius': '20px',
                        'height': 'fit-content',
                        'marginTop': '30px',
                        'marginLeft': '10px',
                        'marginRight': '10px',
                        'boxShadow': '0px 3px 6px rgba(0,0,0,0.1)',
                        'display': 'flex',
                        'flexWrap': 'wrap',
                        'alignItems': 'flex-start',
                        'marginBottom': '30px'
                    }),                    
                    html.Div([
                        html.Div([
                            dcc.Graph(id='grafico-heatmap', config={'displayModeBar': False}, style={'height': '250px', 'width': '100%'}),
                            html.Button(
                                "Resetar Filtro",
                                id="botao-resetar",
                                style={
                                    'fontFamily': 'Aparajita',
                                    'position': 'absolute',
                                    'top': '30px',
                                    'right': '20px',
                                    'zIndex': '1',
                                    'backgroundColor': '#e0e0e0',
                                    'border': '1px solid #ccc',
                                    'padding': '6px 12px',
                                    'borderRadius': '4px',
                                    'cursor': 'pointer',
                                    'fontWeight': 'bold',
                                    'size': '14px'
                                }
                            )
                        ], style={
                            'position': 'relative',
                            'flex': '1',
                            'minWidth': '400px',
                            'maxWidth': '100%',
                        }),

                        html.Div(id='tabela-informacoes-colaboradores', style={
                            'flex': '1',
                            'minWidth': '280px',
                            'maxWidth': '400px',
                            'padding': '20px',
                            'boxSizing': 'border-box',
                            'backgroundColor': "#e0e0e0",
                            'borderRadius': '20px',
                            'height': 'fit-content',
                            'marginTop': '30px',
                            'marginLeft': '240px',
                            'marginRight': '130px',
                            'boxShadow': '0px 3px 6px rgba(0,0,0,0.1)'
                        })
                    ], style={
                        'display': 'flex',
                        'flexWrap': 'wrap',
                        'alignItems': 'flex-start',
                        'marginBottom': '30px'
                    }),

                    html.Div([
                        html.Div([dcc.Graph(id='grafico-dispersao', style={'height': '550px', 'width': '100%'})], style={'flex': '1', 'marginRight': '20px'}),
                        html.Div([dcc.Graph(id='grafico-presenca-cargo', style={'height': '550px', 'width': '100%'})], style={'flex': '1'})
                    ], style={'display': 'flex', 'flexWrap': 'wrap'}),

                    html.Div([
                        html.Div([dcc.Graph(id='grafico-boxplot', style={'height': '600px', 'width': '100%'})], style={'flex': '2'})
                    ], style={'display': 'flex', 'flexWrap': 'wrap'}),

                    html.Div([
                                html.Button("▼", id="toggle-resumo-btn", n_clicks=0, style={
                                    'fontSize': '18px',
                                    'marginLeft': '10px',
                                    'background': 'none',
                                    'border': 'none',
                                    'cursor': 'pointer',
                                    'verticalAlign': 'middle'
                                }),
                                html.Span("Resumo Específico por Área (Mensalistas/Executivos)", style={
                                    'fontWeight': 'bold',
                                    'fontSize': '18px',
                                    'color': '#000000'
                                })
                            ], style={'marginBottom': '2px'}),

                    html.Div(id='resumo-container', style={'marginBottom': '20px'}),
                        
                    html.Div([
                        html.H3("Presença dos Colaboradores", style={'marginBottom': '18px', 'marginTop': '18px', 'color': '#000000'}),
                        
                        dcc.Dropdown(
                            id='dropdown-chapa',
                            options=[],
                            value=None,
                            placeholder='Buscar colaborador pela chapa...',
                            clearable=True,
                            searchable=True,
                            style={'width': '300px', 'marginTop': '10px', 'marginBottom': '10px'}
                        ),

                        html.Div(id='tabela-detalhada-colaboradores')
                    ]),

                    dcc.Store(id='faixa-selecionada', data=None),
                    dcc.Store(id='area-selecionada', data='GERAL'),
                    dcc.Store(id='subarea-selecionada', data=None),
                    dcc.Store(id='ultimo-clique', data={'faixa': None, 'timestamp': 0}),
                ], style={'padding': '20px', 'backgroundColor': 'white'})
            ])
        )
    ),
    # --- Página Analítica por Pessoa ---
    dcc.Loading(
        id="loading-individual",
        type="circle",
        fullscreen=True,
        children=html.Div(
            id='conteudo-individual',
            children=[
                html.Button(
                    "Voltar",
                    id='botao-voltar',
                    n_clicks=0,
                    style={
                        'color': 'Black',
                        'fontFamily': 'Aparajita',
                        'fontWeight': 'bold',
                        'marginTop': '20px',
                        'display': 'block',
                        'marginLeft': '10px',
                        'marginRight': 'auto',
                        'padding': '10px 20px',
                        'fontSize': '14px',
                        'backgroundColor': '#e0e0e0',
                        'border': 'none',
                        'borderRadius': '5px',
                        'cursor': 'pointer',
                        'boxShadow': '0px 2px 6px rgba(0, 0, 0, 0.25)'
                    }
                )
            ],style={'display': 'none', 'backgroundColor': 'white'}
        )
    ),
    # --- Página Analítica por Área/Subárea ---
    dcc.Loading(
        id="loading-analitico-area",
        type="circle",
        fullscreen=True,
        children=html.Div(
            id='conteudo-analitico-area',
            children=[
                html.Button(
                    "Voltar",
                    id='botao-voltar-area',
                    n_clicks=0,
                    style={
                        'color': 'Black','fontFamily': 'Aparajita','fontWeight': 'bold',
                        'marginTop': '20px','display': 'block','marginLeft': '10px',
                        'marginRight': 'auto','padding': '10px 20px','fontSize': '14px',
                        'backgroundColor': '#e0e0e0','border': 'none','borderRadius': '5px',
                        'cursor': 'pointer','boxShadow': '0px 2px 6px rgba(0,0,0,0.25)'
                    }
                )
            ],
            style={'display': 'none', 'backgroundColor': 'white'}
        )
    )
])

@app.callback(
    Output('resumo-container', 'style'),
    Output('toggle-resumo-btn', 'children'),
    Input('toggle-resumo-btn', 'n_clicks'),
    State('resumo-container', 'style')
)
def toggle_resumo(n_clicks, style):
    if not n_clicks:
        return estilo_oculto, "►"
    exibir = style.get('display') == 'none'
    return {'display': 'block' if exibir else 'none'}, "▼" if exibir else "►"

@app.callback(
    Output('dropdown-subarea', 'options'),
    Output('dropdown-subarea', 'style'),
    Output('dropdown-subarea', 'value'),
    Input('dropdown-area', 'value')
)
def atualizar_subdropdown(area):
    if not area or area == 'GERAL':
        return [], estilo_oculto, None

    heranca = her[her['Area'] == area]['Heranca'].unique()
    subareas_df = her[
        (her['Area'].str.startswith(area)) &
        (her['Area'] != area) &
        (her['Heranca'].isin(heranca))
    ]

    if subareas_df.empty:
        return [], estilo_oculto, None

    options = [
        {'label': f"{row['Area']} {row['Departamento']}", 'value': row['Area']}
        for _, row in subareas_df.iterrows()
    ]
    return options, {'width': '300px', 'marginTop': '10px', 'display': 'block'}, None

@app.callback(
    Output('periodo-slider', 'min'),
    Output('periodo-slider', 'max'),
    Output('periodo-slider', 'value'),
    Input('cenario-toggle', 'value'),
    State('periodo-slider', 'value')
)
def travar_periodo(cenario, valor_atual):
    if cenario == 'c1':
        min_i, max_i = LT_C1_MIN, LT_C1_MAX
    else:
        min_i, max_i = LT_C2_MIN, LT_C2_MAX

    v0, v1 = valor_atual or [min_i, max_i]
    v0 = min(max(v0, min_i), max_i)
    v1 = min(max(v1, min_i), max_i)
    if v0 > v1:
        v0, v1 = min_i, max_i
    return min_i, max_i, [v0, v1]

@app.callback(
    Output('periodo-indices','data'),
    Output('periodo-texto','children'),
    Input('periodo-slider','value')
)
def armazenar_periodo(indices):
    i0, i1 = indices
    d0 = pd.Timestamp(LT_MONTHS[i0])
    d1 = pd.Timestamp(LT_MONTHS[i1]) + pd.offsets.MonthEnd(1)  # fim do mês
    txt = f"Período selecionado: {d0.strftime('%b/%Y')} a {d1.strftime('%b/%Y')}"
    return [i0, i1], txt

@app.callback(
    Output('area-selecionada', 'data'),
    Input('dropdown-area', 'value')
)
@app.callback(
    Output('subarea-selecionada', 'data'),
    Input('dropdown-subarea', 'value')
)
def armazenar_selecao(value):
    return value

@app.callback(
    Output('faixa-selecionada', 'data'),
    Output('ultimo-clique', 'data'),
    Input('grafico-heatmap', 'clickData'),
    Input('botao-resetar', 'n_clicks'),
    State('faixa-selecionada', 'data'),
    State('ultimo-clique', 'data'),
    prevent_initial_call=True
)
def atualizar_faixa(clickData, reset_clicks, faixa_atual, ultimo_clique):
    ctx = callback_context
    trigger = ctx.triggered_id

    faixa_atual = faixa_atual or []
    ultimo_clique = ultimo_clique or {'faixa': None, 'timestamp': 0}

    if trigger == 'grafico-heatmap' and clickData:
        faixa = clickData['points'][0]['x']
        nova_lista = [f for f in faixa_atual if f != faixa] if faixa in faixa_atual else faixa_atual + [faixa]
        return nova_lista, {'faixa': faixa, 'timestamp': time.time()}

    if trigger == 'botao-resetar':
        return [], {'faixa': None, 'timestamp': 0}

    return faixa_atual, ultimo_clique

def aplicar_filtro_faixas(df, coluna, faixas):
    if not faixas or not all(" a " in f for f in faixas):
        return df
    return df[df[coluna].apply(lambda x: esta_em_alguma_faixa(x, faixas))]

def aplicar_filtro_por_area(df, area, subarea, her_df):
    if area and area != 'GERAL':
        if subarea:
            departamentos = her_df[her_df['Area'] == subarea]['Heranca'].unique()
            return df[(df['Area'] == subarea) & (df['Heranca'].isin(departamentos))]
        else:
            departamentos = her_df[her_df['Area'] == area]['Heranca'].unique()
            return df[df['Area'].str.startswith(area) & df['Heranca'].isin(departamentos)]
    return df

def esta_em_alguma_faixa(valor, faixas):
    for faixa in faixas:
        try:
            faixa_min, faixa_max = map(float, faixa.split(" a "))
            if faixa_min <= valor < faixa_max:
                return True
        except:
            continue
    return False

def interpolar_cor_hex(cor1, cor2, t):
    c1 = [int(cor1[i:i + 2], 16) for i in (1, 3, 5)]
    c2 = [int(cor2[i:i + 2], 16) for i in (1, 3, 5)]
    interpolada = [int(c1[j] + (c2[j] - c1[j]) * t) for j in range(3)]
    return f"#{interpolada[0]:02x}{interpolada[1]:02x}{interpolada[2]:02x}"

def normalizar_str(s):
    return str(s).strip().replace("_", " ").replace("-", "/")

def slug(s):  # seguro para URL
    return str(s).replace(" ", "_").replace("/", "-")

@app.callback(
    Output('tabela-detalhada-colaboradores', 'children'),
    Output('dropdown-chapa', 'options'),
    Output('grafico-heatmap', 'figure'),
    Output('grafico-dispersao', 'figure'),
    Output('grafico-presenca-cargo', 'figure'),
    Output('grafico-boxplot', 'figure'),
    Output('tabela-informacoes-colaboradores', 'children'),
    Output('resumo-container', 'children'),
    Input('afastados-toggle', 'value'),
    Input('viajam-toggle', 'value'),
    Input('dropdown-chapa', 'value'),
    Input('faixa-selecionada', 'data'),
    Input('area-selecionada', 'data'),
    Input('subarea-selecionada', 'data')
)
def atualizar_tudo(afastados, viajam, chapa_selecionada, faixa_selecionada, area_selecionada, subarea_selecionada):
    # Normaliza faixas em lista
    if not faixa_selecionada:
        faixas = []
    elif isinstance(faixa_selecionada, str):
        faixas = [faixa_selecionada]
    else:
        faixas = faixa_selecionada

    # Base filtrada
    if afastados == 'right':
        df_filtrado = df_semana_merged[df_semana_merged['Afastado'] != 'Sim'].copy()
    else:
        df_filtrado = df_semana_merged.copy()

    # Mantém encadeamento de filtros (não reinicia do df_semana_merged)
    if viajam == 'right':
        # Se houver alguma lógica de 'áreas que viajam', substitua o predicado abaixo.
        df_filtrado = df_filtrado[~df_filtrado['Area'].isin(['', ''])]

    # Filtra por área / subárea
    df_filtrado = aplicar_filtro_por_area(df_filtrado, area_selecionada, subarea_selecionada, her)

    # --- Parte 1: Tabela detalhada colaboradores ---
    media_colab = df_filtrado.groupby(['Area', 'Departamento', 'Cadastro', 'Nome', 'Cargo Especifico'])['Contagem Presença'].mean().reset_index()
    media_colab.columns = ['Área', 'Departamento', 'Chapa', 'Nome', 'Cargo', 'Média Dia']
    media_colab['Média Dia'] = media_colab['Média Dia'].round(2)

    if chapa_selecionada:
        media_colab = media_colab[media_colab['Chapa'] == chapa_selecionada]

    # Filtro por faixa
    media_colab = aplicar_filtro_faixas(media_colab, 'Média Dia', faixas)

    media_colab['Média %'] = (((media_colab['Média Dia'] / 2) * 100) - 100).round(2)
    media_colab = media_colab.sort_values(by=['Área', 'Média %'], ascending=True)
    media_colab['Média %'] = media_colab['Média %'].astype(str) + ' %'

    tabela = html.Div([
        html.Table([
            html.Thead([
                html.Tr([
                    html.Th(col, style={
                        'textAlign': 'left',
                        'backgroundColor': '#f0f0f0',
                        'padding': '10px',
                        'fontWeight': 'bold',
                        'borderBottom': '2px solid #ccc',
                        'borderRight': '1px solid #ccc'
                    }) for col in media_colab.columns
                ])
            ]),
            html.Tbody([
                html.Tr([
                    html.Td(
                        dcc.Link(str(valor), href=f"/pessoa/{linha[2]}", style={
                            'display': 'block',
                            'width': '100%',
                            'height': '100%',
                            'color': (
                                'green' if (
                                    isinstance(linha[5], (int, float)) and linha[5] >= 2 or
                                    (isinstance(linha[6], str) and '%' in linha[6] and float(linha[6].replace('%', '').strip()) >= 0)
                                ) else 'red'
                            ),
                            'textDecoration': 'none',
                        }),
                        style={
                            'textAlign': 'left',
                            'padding': '8px',
                            'borderBottom': '1px solid #eee',
                            'borderRight': '1px solid #eee' if i < len(media_colab.columns) - 1 else 'none',
                            'backgroundColor': 'white'
                        }
                    ) for i, valor in enumerate(linha)
                ]) for linha in media_colab.values
            ])
        ], style={
            'borderCollapse': 'separate',
            'borderSpacing': '0',
            'width': '100%',
            'fontSize': '15px',
            'boxShadow': '0 4px 12px rgba(0, 0, 0, 0.05)',
            'borderRadius': '12px',
            'overflow': 'hidden'
        })
    ])

    # --- Parte 2: Opções dropdown chapa ---
    media_presenca = df_filtrado.groupby(['Cadastro', 'Nome'])['Contagem Presença'].mean().reset_index()
    media_presenca.columns = ['Chapa', 'Nome', 'Média']
    media_presenca = aplicar_filtro_faixas(media_presenca, 'Média', faixas)
    media_presenca = media_presenca.sort_values(by='Nome')
    opcoes = [{'label': f"{row['Chapa']} - {row['Nome']}", 'value': row['Chapa']} for _, row in media_presenca.iterrows()]

    # --- Parte 3: Indicadores e gráficos ---
    total_colaboradores = df_filtrado['Cadastro'].nunique()
    total_mensalistas = df_filtrado[df_filtrado['Cargo'] == 'M']['Cadastro'].nunique()
    total_executivos = df_filtrado[df_filtrado['Cargo'] == 'E']['Cadastro'].nunique()
    media_presenca_geral = round(df_filtrado['Contagem Presença'].mean(), 2)
    media_presenca_mensalistas = round(df_filtrado[df_filtrado['Cargo'] == 'M']['Contagem Presença'].mean(), 2)
    media_presenca_executivos = round(df_filtrado[df_filtrado['Cargo'] == 'E']['Contagem Presença'].mean(), 2)

    tabela_info = html.Table([
        html.Thead([html.Tr([html.Th("Informações de Colaboradores", colSpan=2)], style={'fontSize': '20px'})]),
        html.Tbody([
            html.Tr([html.Td("Total de Colaboradores:"), html.Td(total_colaboradores)]),
            html.Tr([html.Td("Mensalistas:"), html.Td(total_mensalistas)]),
            html.Tr([html.Td("Executivos:"), html.Td(total_executivos)]),
            html.Tr([html.Td(html.Br()), html.Td()]),
            html.Tr([html.Th("Média(%) de Presença por Semana", colSpan=2)], style={'fontSize': '20px'}),
            html.Tr([html.Td("Total:"), html.Td(f"{media_presenca_geral}%")]),
            html.Tr([html.Td("Mensalistas:"), html.Td(f"{media_presenca_mensalistas}%")]),
            html.Tr([html.Td("Executivos:"), html.Td(f"{media_presenca_executivos}%")])
        ])
    ], style={'fontSize': '16px', 'width': '100%'})

    # Média por pessoa (base para gráficos) — já está agregada, então reuso ao invés de reagrupar
    media_presenca_detalhada = df_filtrado.groupby('Cadastro')['Contagem Presença'].mean().reset_index()
    media_presenca_detalhada.columns = ['Cadastro', 'Média Presença']
    media_presenca_detalhada = media_presenca_detalhada.merge(
        df_filtrado[['Cadastro', 'Cargo', 'Nome', 'Area', 'Departamento']].drop_duplicates(),
        on='Cadastro'
    )

    # ===== Heatmap (faixas) =====
    media_por_pessoa = media_presenca_detalhada[['Cadastro', 'Média Presença']].copy()
    # limita faixas de 0 a 7 e joga o 7 em "6 a 7"
    bins = list(range(0, 8))  # 0,1,2,3,4,5,6,7  -> faixas: 0-1, 1-2, ..., 6-7
    labels_bins = [f"{i} a {i+1}" for i in range(7)]
    media_por_pessoa['Média Presença'] = media_por_pessoa['Média Presença'].clip(upper=7)
    media_por_pessoa['Faixa'] = pd.cut(
        media_por_pessoa['Média Presença'],
        bins=bins,
        labels=labels_bins,
        right=True,           # usa (a, b]  -> 7 entra em (6,7]
        include_lowest=True   # inclui 0 em [0,1]
    )

    dados = media_por_pessoa[media_por_pessoa['Faixa'].isin(faixas)] if faixas else media_por_pessoa
    contagem_faixa = dados['Faixa'].value_counts().sort_index()
    x_faixas = sorted(contagem_faixa.index.tolist(), key=lambda x: int(x.split(" ")[0]))
    y_faixas = contagem_faixa.reindex(x_faixas, fill_value=0).values.tolist()

    num_faixas = len(x_faixas)
    colorscale = [None] * num_faixas

    quant_vermelho, indices_vermelho = [], []
    quant_amarelo, indices_amarelo = [], []
    quant_verde, indices_verde = [], []

    for i, faixa in enumerate(x_faixas):
        faixa_min_val = int(faixa.split(" ")[0])
        qtd = y_faixas[i]
        if faixa_min_val < 2:
            quant_vermelho.append(qtd); indices_vermelho.append(i)
        elif faixa_min_val > 4:
            quant_amarelo.append(qtd); indices_amarelo.append(i)
        else:
            quant_verde.append(qtd); indices_verde.append(i)

    min_vermelho, max_vermelho = (min(quant_vermelho), max(quant_vermelho)) if quant_vermelho else (0, 1)
    min_amarelo, max_amarelo = (min(quant_amarelo), max(quant_amarelo)) if quant_amarelo else (0, 1)
    min_verde, max_verde = (min(quant_verde), max(quant_verde)) if quant_verde else (0, 1)

    for idx in indices_vermelho:
        qtd = y_faixas[idx]
        t = (qtd - min_vermelho) / (max_vermelho - min_vermelho) if max_vermelho > min_vermelho else 0
        pos = idx / (num_faixas - 1) if num_faixas > 1 else 0
        colorscale[idx] = [pos, interpolar_cor_hex("#fc9898", "#ef6464", t)]

    for idx in indices_amarelo:
        qtd = y_faixas[idx]
        t = (qtd - min_amarelo) / (max_amarelo - min_amarelo) if max_amarelo > min_amarelo else 0
        pos = idx / (num_faixas - 1) if num_faixas > 1 else 0
        colorscale[idx] = [pos, interpolar_cor_hex("#fff9a9", "#ffef09", t)]

    for idx in indices_verde:
        qtd = y_faixas[idx]
        t = (qtd - min_verde) / (max_verde - min_verde) if max_verde > min_verde else 0
        pos = idx / (num_faixas - 1) if num_faixas > 1 else 0
        colorscale[idx] = [pos, interpolar_cor_hex("#9effd0", "#3fbe81", t)]

    for i in range(num_faixas):
        if colorscale[i] is None:
            pos = i / (num_faixas - 1) if num_faixas > 1 else 0
            colorscale[i] = [pos, "#cccccc"]

    fig_heat = go.Figure(data=go.Heatmap(
        z=[list(range(num_faixas))],
        x=x_faixas,
        y=["Distribuição"],
        colorscale=colorscale,
        hovertemplate="Faixa: %{x}<br>Qtd Pessoas: %{customdata}<extra></extra>",
        customdata=[y_faixas],
        showscale=False
    ))

    annotations = [
        dict(
            x=x_faixas[i],
            y="Distribuição",
            text=str(val),
            showarrow=False,
            font=dict(color="black", size=15, family="Aparajita"),
            xanchor="center",
            yanchor="middle"
        ) for i, val in enumerate(y_faixas)
    ]

    fig_heat.update_layout(
        title={
            'text': "Distribuição de Colaboradores por Faixa",
            'font': {'family': 'Aparajita', 'color': '#000000', 'size': 24},
        },
        xaxis_title="Faixa de Média",
        yaxis_title="",
        xaxis=dict(type='category'),
        yaxis=dict(showticklabels=False),
        height=250,
        showlegend=False,
        annotations=annotations
    )

    # Calcular média por colaborador (para boxplot/área)
    medias_individuais = df_filtrado.groupby(['Cadastro', 'Nome', 'Area'])['Contagem Presença'].mean().reset_index().round(2)
    medias_individuais.columns = ['Chapa', 'Nome', 'Área', 'Média Presença']
    medias_individuais['Média Presença'] = medias_individuais['Média Presença'].round(2)

    if area_selecionada and area_selecionada != 'GERAL':
        if subarea_selecionada:
            medias_individuais = medias_individuais[medias_individuais['Área'] == subarea_selecionada]
        else:
            medias_individuais = medias_individuais[medias_individuais['Área'].str.startswith(area_selecionada)]
        
    # Tabela resumo área/departamento
    media_colab_resumo = df_filtrado.groupby(['Area', 'Departamento', 'Cadastro', 'Nome'])['Contagem Presença'].mean().reset_index()
    media_colab_resumo.columns = ['Área', 'Departamento', 'Chapa', 'Nome', 'Média Dia']
    media_colab_resumo['Média Dia'] = media_colab_resumo['Média Dia'].round(2)

    # Filtro por faixa
    medias_individuais = aplicar_filtro_faixas(medias_individuais, 'Média Presença', faixas)

    fig_boxplot = go.Figure()
    for area_nome in sorted(medias_individuais['Área'].unique()):
        dados_area = medias_individuais[medias_individuais['Área'] == area_nome]
        fig_boxplot.add_trace(go.Box(
            y=dados_area['Média Presença'],
            name=area_nome,
            boxpoints='outliers',
            marker_color='#1f77b4',
            line_color='#1f77b4',
            fillcolor='#a4c6eb',
            hovertemplate=area_nome + "<br>Média: %{y:.2f}<extra></extra>"
        ))

    fig_boxplot.update_layout(
        title={
            'text': f"Boxplot de Presença por Área ({', '.join(faixas) if faixas else 'Todos'})",
            'font': {'family': 'Aparajita', 'size': 24}
        },
        yaxis_title="Média de Presença Semanal",
        xaxis=dict(
            title="",
            rangeslider=dict(visible=True),
            showline=True,
            zeroline=True
        ),
        boxmode='group',
        height=600,
        shapes=[
            dict(
                type="line",
                xref="paper",
                yref="y",
                x0=0,
                x1=1,
                y0=2,
                y1=2,
                line=dict(
                    color="red",
                    width=2,
                    dash="solid"
                )
            )
        ]
    )

    # Dispersões
    media_presenca_detalhada_f = aplicar_filtro_faixas(media_presenca_detalhada, 'Média Presença', faixas)
    media_presenca_detalhada_f['Média Presença Arred'] = media_presenca_detalhada_f['Média Presença'].round(1)
    dados_disp = media_presenca_detalhada_f.groupby('Média Presença Arred')['Cadastro'].count().reset_index()
    dados_disp.columns = ['Média Presença', 'Qtd Pessoas']

    fig_dispersao = go.Figure()
    fig_dispersao.add_trace(go.Scatter(
        x=dados_disp['Média Presença'],
        y=dados_disp['Qtd Pessoas'],
        mode='markers+text',
        marker=dict(size=12, color="#0003b4"),
        hovertemplate="Qtd Pessoas: %{y}<br>Média Presença: %{x:.1f}<extra></extra>"
    ))
    fig_dispersao.update_layout(
        title={'text': f"Dispersão Geral ({', '.join(faixas) if faixas else 'Todos'})", 'font': {'family': 'Aparajita', 'color': '#000000', 'size': 24}},
        xaxis_title="Média de Presença Semanal",
        yaxis_title="Qtd de Pessoas",
        height=500
    )

    fig_cargo = go.Figure()
    for cod, nome, cor in [('M', 'Mensalistas', '#7cc3e9'), ('E', 'Executivos', "#1574b3")]:
        df_aux = media_presenca_detalhada_f[media_presenca_detalhada_f['Cargo'] == cod].copy()
        df_aux['Média Presença Arred'] = df_aux['Média Presença'].round(1)
        df_cargo = df_aux.groupby('Média Presença Arred')['Cadastro'].count().reset_index()
        df_cargo.columns = ['Média Presença', 'Qtd Pessoas']

        fig_cargo.add_trace(go.Scatter(
            x=df_cargo['Média Presença'],
            y=df_cargo['Qtd Pessoas'],
            mode='markers+text',
            name=nome,
            marker=dict(size=12, color=cor),
            hovertemplate=(nome + "<br>Qtd Pessoas: %{y}<br>Média Presença: %{x:.1f}<extra></extra>")
        ))

    fig_cargo.update_layout(
        title={'text': f"Presença por Cargo ({', '.join(faixas) if faixas else 'Todos'})", 'font': {'family': 'Aparajita', 'color': '#000000', 'size': 24}},
        xaxis_title="Média de Presença Semanal",
        yaxis_title="Qtd de Pessoas",
        height=500,
        legend=dict(
            x=1,
            y=0.87,
            traceorder='normal',
            orientation='v',
            xanchor='right',
            yanchor='bottom',
            font=dict(size=12),
            bgcolor='rgba(255,255,255,0.7)'
        )
    )

    media_colab_resumo = media_colab_resumo.merge(df_filtrado[['Cadastro', 'Cargo']].drop_duplicates(), how='left', left_on='Chapa', right_on='Cadastro')

    resumo_area = media_colab_resumo.groupby(['Área', 'Departamento']).agg(
        Qtd_Pessoas=('Chapa', 'count'),
        Qtd_Mensalistas=('Cargo', lambda x: (x == 'M').sum()),
        Qtd_Executivos=('Cargo', lambda x: (x != 'M').sum()),
        Media_Dia=('Média Dia', 'mean')
    ).reset_index()

    resumo_area['Média Dia'] = resumo_area['Media_Dia'].round(2)
    resumo_area['Média %'] = ((resumo_area['Média Dia'] / 2) * 100 - 100).round(2)

    # Filtro por faixa
    resumo_area = aplicar_filtro_faixas(resumo_area, 'Média Dia', faixas)

    # dentro de atualizar_tudo -> _td_link_area (só trocar o href):
    def _td_link_area(texto, area, extra_style=None):
        base_style = {
            'display': 'block', 'width': '100%', 'height': '100%',
            'color': 'inherit', 'textDecoration': 'none', 'padding': '8px'
        }
        if extra_style:
            base_style.update(extra_style)
        # >>> use slug aqui, não normalizar_str
        return dcc.Link(str(texto), href=f"/area/{slug(area)}", style=base_style)

    def td_style(align='left', width='auto'):
        return {
            'borderBottom': '1px solid #eee',
            'borderRight': '1px solid #eee',
            'cursor': 'pointer',
            'textAlign': align,
            'width': width,
            'padding': '5px 10px'  # 5px vertical, 10px horizontal
        }

    def th_style(align='left', width='auto'):
        return {
            'padding': '10px 10px',  # igual para o cabeçalho
            'backgroundColor': '#f0f0f0',
            'textAlign': align,
            'fontWeight': 'bold',
            'borderBottom': '2px solid #ccc',
            'borderRight': '1px solid #ccc',
            'width': width
        }

    tabela_resumo_area = html.Div([
        html.Div(children=[
            html.Table([
                html.Thead([
                    html.Tr([
                        html.Th('Área', style=th_style('left', '0%')),
                        html.Th('Departamento', style=th_style('left', '0%')),
                        html.Th('Mensalistas', style=th_style('left', '0%')),
                        html.Th('Executivos', style=th_style('left', '0%')),
                        html.Th('Total', style=th_style('left', '0%')),
                        html.Th('Média Dia', style=th_style('left', '0%')),
                        html.Th('Média %', style=th_style('left', '0%')),
                    ])
                ]),
                html.Tbody([
                    html.Tr([
                        html.Td(_td_link_area(row['Área'], row['Área']),
                                style=td_style('left', '0%')),
                        html.Td(_td_link_area(row['Departamento'], row['Área']),
                                style=td_style('left', '0%')),
                        html.Td(_td_link_area(row['Qtd_Mensalistas'], row['Área']),
                                style=td_style('left', '0%')),
                        html.Td(_td_link_area(row['Qtd_Executivos'], row['Área']),
                                style=td_style('left', '0%')),
                        html.Td(_td_link_area(row['Qtd_Pessoas'], row['Área']),
                                style=td_style('left', '0%')),
                        html.Td(_td_link_area(row['Média Dia'], row['Área']),
                                style=td_style('left', '0%')),
                        html.Td(_td_link_area(f"{row['Média %']} %", row['Área']),
                                style=td_style('left', '0%')),
                    ],
                    style={'color':'red'} if row['Média Dia'] < 2 else {'color':'green'})
                    for _, row in resumo_area.iterrows()
                ])
            ], style={
                'borderCollapse': 'collapse',
                'width': '100%',
                'fontSize': '15px',
                'boxShadow': '0 4px 12px rgba(0, 0, 0, 0.05)',
                'borderRadius': '12px',
                'overflow': 'hidden'
            })
        ])
    ])

    return tabela, opcoes, fig_heat, fig_dispersao, fig_cargo, fig_boxplot, tabela_info, tabela_resumo_area

def fig_to_base64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode("utf-8")
    plt.close(fig)
    return f"data:image/png;base64,{encoded}"

def obter_periodo_admissao(chapa):
    pessoa = afastados[afastados['Cadastro'] == chapa]
    if not pessoa.empty:
        adm = pessoa['Data_Adm'].values[0]
        dem = pessoa['Data_Dem'].values[0] if pd.notna(pessoa['Data_Dem'].values[0]) else pd.Timestamp.max
    else:
        pessoa = departamentos[departamentos['Cadastro'] == chapa]
        if not pessoa.empty:
            adm = pd.to_datetime(pessoa['Data_Admissão'].values[0], dayfirst=True)
            dem = pd.Timestamp.max
        else:
            return None, None
    return adm, dem

def calcular_percentual_presenca(dias_presenca_por_semana):
    pontuacao_semanal = dias_presenca_por_semana.apply(lambda x: 1 if x >= 2 else (0.5 if x == 1 else 0))
    return round(pontuacao_semanal.mean() * 100, 1)

def gerar_relatorio_formatado(nome_input, chapa_input, media_presenca_semana, percentuais_presenca):
    return (f"   Relatório Corporativo - Sistema Integrado de Gestão de Frequência da Volkswagen\n\n"
            f"   Este relatório tem como finalidade fornecer uma base de dados consolidada, com visualizações estratégicas e indicadores de performance "
            f"relacionados à presença e ao regime de trabalho dos colaboradores entre janeiro de 2024 e março de 2025.\n\n"
            rf"  O colaborador {nome_input} ({chapa_input}) apresentou uma média semanal de presença de "
            rf"$\bf{{{media_presenca_semana}}}$ dia(s) no formato presencial. Para fins de cálculo, foi adotada uma metodologia de pontuação semanal baseada na quantidade de dias presenciais registrados:"
            f"\n- 0 dias presenciais na semana = 0% de presença;\n"
            f"- 1 dia presencial = 50%;\n"
            f"- 2 ou mais dias presenciais = 100%.\n"
            r"Ao longo do período analisado, a presença semanal média do colaborador no formato presencial foi de "
            rf"$\bf{{{percentuais_presenca}\%}}$."
            f"\n\n   Importante destacar que, para este relatório, não foram consideradas como justificativas válidas de ausência: feriados, férias individuais, "
            f"viagens a trabalho ou ausências previamente acordadas. A única exceção aplicada foi para períodos de férias coletivas, devidamente identificadas "
            f"e excluídas do cálculo para evitar distorções. "
            f'Foram identificadas as seguintes semanas com presença muito baixa, ou seja, entre "0" e "1" dia durante a semana (Ano-Semana):'
    )

@app.callback(
    Output('conteudo-individual', 'children'),
    Output('conteudo-individual', 'style'),
    Output('pagina-geral', 'style', allow_duplicate=True),
    Input('url', 'pathname'),
    Input('botao-voltar', 'n_clicks'),
    prevent_initial_call=True
)
def exibir_conteudo_individual(pathname, n_clicks):
    if n_clicks is not None and n_clicks > 0:
        # Volta para a tela inicial:
        return None, estilo_oculto , estilo_exibir 

    # Verifica se o pathname contém '/pessoa/'
    if '/pessoa/' in pathname:
        chapa_input = pathname.split('/pessoa/')[1]
    else:
        return None, estilo_oculto , estilo_exibir 

    if not chapa_input:
        return None, estilo_oculto , estilo_exibir 

    if chapa_input in df_semana_merged['Cadastro'].values:
        # Filtra dados por chapa informada:
        df_chapa_semana = df_semana_merged[df_semana_merged['Cadastro'] == chapa_input]
        df_chapa_data = df_datas[df_datas['Cadastro'] == chapa_input]
        
        # Extrai o nome formatado:
        nome_input = df_chapa_semana['Nome'].values[0].title()

    # Recria a coluna Semana_Ano com todos os dados da pessoa:
    df_chapa_semana['Semana_Ano'] = (
        df_chapa_semana['Ano'].astype(str) + "-" + df_chapa_semana['Semana por Ano'].astype(str).str.zfill(2)
    )

    # Contar presenças por semana:
    dias_presenca_por_semana = df_chapa_semana.set_index('Semana_Ano')['Contagem Presença']

    # Calcular média de presença:
    media_presenca_semana = round(dias_presenca_por_semana.mean(), 2)

    # Calcular o percentual em relação a média:
    percentual_da_media = round(((media_presenca_semana / 2) * 100) - 100, 2)

    # Calcular percentual de presença:
    percentuais_presenca = calcular_percentual_presenca(dias_presenca_por_semana)

    # Filtrar DataFrames para presença:
    if chapa_input:
        df_filtrado_presenca = df_presenca_merged_por_cadastro[df_presenca_merged_por_cadastro['Cadastro'].astype(str) == chapa_input]
    else:
        df_filtrado_presenca = df_presenca_merged_por_cadastro

    # Contagem de presenças:
    distribuicao_presencas = df_filtrado_presenca['Contagem Presença'].value_counts().sort_index()

    # Aplica um índice único:
    x = sorted(set(distribuicao_presencas.index))

    # Preenche contagens reindexadas:
    y_presencas = distribuicao_presencas.reindex(x, fill_value=0)

    # Ordena as semanas para garantir sequência cronológica:
    dias_presenca_por_semana_ordenado = dias_presenca_por_semana.sort_index()

    # Cria os vetores numéricos:
    x_semanas = list(range(1, len(dias_presenca_por_semana_ordenado) + 1))
    y_presenca_por_semana = dias_presenca_por_semana_ordenado.values

    # Preparar datas do heatmap:
    df_chapa_data['Data'] = pd.to_datetime(df_chapa_data['Data'])

    # Obter o período de admissão e demissão usando a função auxiliar:
    data_adm, data_dem = obter_periodo_admissao(chapa_input)

    # Filtrar semanas somente se datas válidas:
    if data_adm is not None and data_dem is not None:
        todas_semanas_filtradas = [
            semana for semana in todas_semanas
            if data_adm <= semana <= data_dem
        ]
    else:
        todas_semanas_filtradas = []

    # Construir datas expandidas:
    datas_expandidas = []
    for data in pd.to_datetime(todas_semanas_filtradas):
        inicio_semana = data - pd.to_timedelta(data.weekday(), unit="d")
        semana_completa = [inicio_semana + pd.Timedelta(days=i) for i in range(7)]
        datas_expandidas.extend(semana_completa)
        
    # DataFrame com todas as datas:
    df_todas_datas = pd.DataFrame({"Data": datas_expandidas})

    # Extração de componentes da data:
    df_todas_datas["Ano"] = df_todas_datas["Data"].dt.isocalendar().year
    df_todas_datas["Semana_Num"] = df_todas_datas["Data"].dt.isocalendar().week
    df_todas_datas["Início da Semana"] = df_todas_datas["Data"] - pd.to_timedelta(df_todas_datas["Data"].dt.weekday, unit="d")
    df_todas_datas["Fim da Semana"] = df_todas_datas["Início da Semana"] + pd.to_timedelta(6, unit="d")

    # Cria label da semana:
    df_todas_datas["Semana_Label"] = (
        df_todas_datas["Ano"].astype(str)
        + "-" + df_todas_datas["Semana_Num"].astype(str).str.zfill(2)
        + " | "
        + df_todas_datas["Início da Semana"].dt.strftime('%d/%m/%Y')
        + "-"
        + df_todas_datas["Fim da Semana"].dt.strftime('%d/%m/%Y')
    )

    # Mapeia nomes dos dias:
    df_todas_datas["Dia da Semana"] = df_todas_datas["Data"].dt.day_name().map(dias_portugues)

    # Merge com presenças:
    df_merged = df_todas_datas.merge(
        df_chapa_data[["Data", "Cadastro"]],
        how="left",
        on="Data"
    )

    # Pivot Table do heatmap:
    tabela_heatmap = df_merged.pivot_table(
        index="Semana_Label",
        columns="Dia da Semana",
        values="Cadastro",
        aggfunc="count",
        fill_value=0
    )

    # Ordena colunas dos dias:
    tabela_heatmap = tabela_heatmap.reindex(columns=ordenacao_dias)

    # Define colormap:
    cmap_personalizado = ListedColormap(["#ffb3b3", '#f4cccc', '#e3dacd', '#dae5d1', '#d9ead3'])

    # Cria figura geral:
    fig = plt.figure(figsize=(28, 45))

    # Define grade principal:
    gs = gridspec.GridSpec(
        3, 2,
        height_ratios=[0.7, 0.5, 3.0],
        width_ratios=[1.9, 1.9]
    )

    # Subgrid para heatmaps:
    gs_heatmaps = gridspec.GridSpecFromSubplotSpec(
        2, 1,
        subplot_spec=gs[2,0],
        height_ratios=[tabela_heatmap.shape[0], 1],
        hspace=0.02
    )

    # Subtítulo colaborador:
    fig.suptitle(
        f"Colaborador: {nome_input}\nChapa: {chapa_input}",
        x=0.01,
        y=1.012,
        fontsize=30,
        fontweight='bold',
        ha='left'
    )

    # Linha pontilhada horizontal:
    linha = mlines.Line2D(
        [0.01, 0.99],
        [0.992, 0.992],
        color='gray',
        linestyle='dashed',
        linewidth=1.5,
        alpha=0.6,
        transform=fig.transFigure,
        figure=fig
    )
    fig.lines.append(linha)

    # Título principal:
    fig.text(
        0.5,
        0.953,
        "Relatório de Frequência e Regime de Trabalho no Período de Janeiro de 2024 a Março de 2025\n\n\n",
        fontsize=28,
        ha='center'
    )

    #~~~~~~~~~~~~~~~~~~~~~ Subplot 1 - Clock Demo Style Gauge ~~~~~~~~~~~~~~~~~~~~~#
    ax_clock = fig.add_subplot(gs[0, 0], polar=True)
    ax_clock.set_theta_offset(np.pi)  # Começar da esquerda...
    ax_clock.set_theta_direction(-1)  # Girar sentido horário...

    # Parâmetros:
    current_val = media_presenca_semana

    # Limpa marcações:
    ax_clock.set_yticklabels([])
    ax_clock.set_xticklabels([])
    ax_clock.set_axis_off()

    # Define cores de zonas:
    colors = [
        '#ffb3b3', '#f4cccc', '#d9ead3', '#b6e6c5',
        '#96ddb1', '#6ccf9a', '#3fbe81'
    ]
    bounds = [0, 1, 2, 3, 4, 5, 6, 7]

    for i in range(len(colors)):
        start = np.pi * (bounds[i]/7)
        end = np.pi * (bounds[i+1]/7)
        ax_clock.barh(1, end - start, left=start, height=0.4, color=colors[i], edgecolor='black')

    tick_vals = [0, 1, 2, 3, 4, 5, 6, 7]
    tick_angles = [np.pi * (val / 7) for val in tick_vals]
    tick_labels = [f"{val:.1f}" for val in tick_vals]

    for ang, label in zip(tick_angles, tick_labels):
        ax_clock.text(ang, 1.3, label, ha='center', va='center', fontsize=17, color='gray')

    angle = np.pi * (min(current_val, 7) / 7)
    ax_clock.arrow(angle, 0, 0, 0.95,
                width=0.03,
                head_width=0.08,
                head_length=0.15,
                fc='black', ec='black',
                zorder=5)

    # Texto do valor:
    ax_clock.text(0, 0, f"\n\n\n{current_val:.2f} dias/semana", fontsize=20, ha='center', va='center', color='#444')

    # Inserindo o Retângulo:
    rect = Rectangle(
        (0.264, 0.385),
        0.48,
        0.07,
        linewidth=2,
        edgecolor='black',
        facecolor='none',
        transform=ax_clock.transAxes,
        zorder=10
    )
    ax_clock.add_patch(rect)

    # Título:
    ax_clock.set_title( "\n\nFrequência da Contagem de Presencial\npor Semana (Jan 2024 a Mar 2025)\n", va='bottom', fontsize=20, pad=35)

    # Alerta, se necessário:
    if current_val > 2:
        ax_clock.text(
            0.01, 0,
            f"\n\n\n\n\n⚠️ Acima do necessário! ⚠️\n+{percentual_da_media}%",
            ha='center',
            va='center',
            fontsize=34,
            color='green',
            fontweight='bold',
            path_effects=[patheffects.withStroke(linewidth=3, foreground='white')]
        )
    elif current_val < 2:
        ax_clock.text(
            0.01, 0,
            f"\n\n\n\n\n⚠️ Abaixo do esperado! ⚠️\n{percentual_da_media}%",
            ha='center',
            va='center',
            fontsize=34,
            color='red',
            fontweight='bold',
            path_effects=[patheffects.withStroke(linewidth=3, foreground='white')]
        )

    ax_clock.patch.set_facecolor('white')
    ax_clock.patch.set_alpha(0)

    # Rodapé abaixo do gauge:
    ax_clock.text(
        -0.15, -0.16,
        'Indicador de frequência presencial semanal de "0" a "7" dias úteis por semana.\n'
        'Esse gráfico ajuda a saber a média que o colaborador está presente na planta.',
        ha='left', va='center', fontsize=14, wrap=True, transform=ax_clock.transAxes
    )

    #~~~~~~~~~~~~~~~~~~~~~ Subplot 2 - Linha (distribuição por contagem) ~~~~~~~~~~~~~~~~~~~~~#
    ax_line = fig.add_axes([0.618, 0.8195, 0.35, 0.120])
    ax_line.plot(x, y_presencas, color="#88ff5d", linewidth=4)
    ax_line.fill_between(x, y_presencas, color="#ccffb9", alpha=0.5)
    for xi, yi in zip(x, y_presencas):
        ax_line.text(xi, yi + 0.14, f"{yi}", ha='center', fontsize=17, color="#256b2b")
    ax_line.set_xticks(x)
    ax_line.set_xlabel('Qtd. de Presencial na Semana', fontsize=20)
    ax_line.set_ylabel('Número de Ocorrências', fontsize=20)
    ax_line.set_title(f'Distribuição da Contagem de Presencial\npor Semana (Jan 2024 a Mar 2025)', fontsize=20, pad=25)
    ax_line.grid(True, linestyle='--')
    ax_line.tick_params(axis='x', labelsize=17)
    ax_line.tick_params(axis='y', labelsize=17)
    ax_line.text(
        0, -0.28,
        "Quantidade de semanas para uma certa quantidade de presença.\n"
        "Esse gráfico ajuda a visualizar picos de frequência, transições de regime.",
        ha='left', va='center', fontsize=14, wrap=True, transform=ax_line.transAxes
    )

    #~~~~~~~~~~~~~~~~~~~~~ Subplot 3 - Linha (temporal + tendência) ~~~~~~~~~~~~~~~~~~~~~#
    x_np = np.array(sorted(x_semanas))
    y_np = np.array([y for _, y in sorted(zip(x_semanas, y_presenca_por_semana))])

    # Ajuste polinomial com fallback pra séries curtas
    if len(x_np) >= 2:
        deg = 1 if len(x_np) == 2 else 2
        coef_poly = np.polyfit(x_np, y_np, deg=deg)
        poly_fn = np.poly1d(coef_poly)
        x_pred = np.linspace(x_np[0], x_np[-1], 500)
        inicio = float(poly_fn(x_pred[0])); fim = float(poly_fn(x_pred[-1]))
        cor_tendencia = 'red' if fim < 2 else '#256b2b'
    else:
        x_pred, poly_fn, cor_tendencia = x_np, lambda z: y_np, '#256b2b'

    ax_line = fig.add_axes([0.204, 0.626, 0.680, 0.119])
    ax_line.plot(x_np, y_np, color="#88ff5d", linewidth=4, label='Presencial')
    if len(x_np) >= 2:
        ax_line.plot(x_pred, poly_fn(x_pred), color=cor_tendencia, linestyle='--', linewidth=2, label='Tendência')

    ax_line.set_xlim(x_np[0], x_np[-1] if len(x_np) else 1)

    anos = [label.split("-")[0] for label in dias_presenca_por_semana_ordenado.index]
    semanas = [int(label.split("-")[1]) for label in dias_presenca_por_semana_ordenado.index]

    ax_line.set_xticks(x_semanas)
    ax_line.set_xticklabels(semanas, fontsize=16)

    semanas_por_ano = defaultdict(list)
    for x_val, ano in zip(x_semanas, anos):
        semanas_por_ano[ano].append(x_val)

    for ano, semanas_ano in semanas_por_ano.items():
        centro = np.mean(semanas_ano)
        ax_line.text(
            centro,
            -0.9,
            ano,
            transform=ax_line.transData,
            ha='center',
            va='center',
            fontsize=18,
            color='gray'
        )

    for i in range(1, len(anos)):
        if anos[i] != anos[i - 1]:
            x_val = x_semanas[i] - 0.5
            x_fig, _ = ax_line.transData.transform((x_val, 0))
            x_fig_norm = fig.transFigure.inverted().transform((x_fig, 0))[0]
            bbox = ax_line.get_position()
            y_fig = bbox.y0 - 0.015
            # pequeno segmento vertical (corrige coordenadas idênticas)
            linha_tracejada = Line2D(
                [x_fig_norm, x_fig_norm],
                [y_fig, y_fig + 0.015],
                transform=fig.transFigure,
                linestyle='--',
                color='gray',
                linewidth=2,
                alpha=0.7
            )
            fig.lines.append(linha_tracejada)

    ax_line.set_xlabel('', fontsize=20)
    ax_line.set_ylabel('Quantidade de Presença', fontsize=20)
    ax_line.set_title('Distribuição da Contagem de Presencial em cada Semana (Jan 2024 a Mar 2025)', fontsize=20, pad=25)
    ax_line.grid(True, linestyle='--')
    ax_line.legend(fontsize=16, loc='upper right')
    ax_line.tick_params(axis='x', labelsize=17)
    ax_line.tick_params(axis='y', labelsize=17)

    ax_line.text(
        0, -0.31,
        "Número de presença em cada semana de todo esse período, esse gráfico ajuda a visualizar picos de frequência, transições de regime e a tendência do colaborador.\n\n\n\n\n\n",
        ha='left', va='center', fontsize=14, wrap=True, transform=ax_line.transAxes
    )

    #~~~~~~~~~~~~~~~~~~~~~ Subplot 4 - Heatmap Calendário ~~~~~~~~~~~~~~~~~~~~~#
    ax_heatmap = fig.add_subplot(gs_heatmaps[0])

    sns.heatmap(
        tabela_heatmap,
        cmap=ListedColormap(['#d9ead3']),
        linewidths=1,
        linecolor='gray',
        annot=True,
        annot_kws={"size": 17},
        cbar=False,
        ax=ax_heatmap
    )

    # Pintar semanas com baixa presença:
    for y_idx, semana_label in enumerate(tabela_heatmap.index):
        semana_codigo = semana_label.split(" |")[0].replace(" ", "-")
        if semana_codigo in dias_presenca_por_semana.index:
            qtd_presenca = dias_presenca_por_semana.loc[semana_codigo]
            cor_destaque = "#ffb3b3" if qtd_presenca == 0 else ("#f4cccc" if qtd_presenca == 1 else "#d9ead3")
            for col_idx in range(tabela_heatmap.shape[1]):
                ax_heatmap.add_patch(
                    plt.Rectangle(
                        (col_idx, y_idx),
                        1, 1,
                        facecolor=cor_destaque,
                        edgecolor='gray',
                        linewidth=2,
                        zorder=1
                    )
                )

    ax_heatmap.set_title(f'\n\n\n\n\n\n\nDistribuição da Contagem de Presencial por\nAno e Semana para cada Dia da Semana (Jan 2024 a Mar 2025)\n', fontsize=20, pad=40)
    ax_heatmap.xaxis.set_label_position('top')
    ax_heatmap.xaxis.tick_top()
    ax_heatmap.tick_params(axis='x', bottom=False, labelbottom=False)
    ax_heatmap.set_ylabel('', fontsize=20)
    ax_heatmap.set_xlabel('', fontsize=20)
    ax_heatmap.set_xticklabels(ax_heatmap.get_xticklabels(), fontsize=20)
    ax_heatmap.set_yticklabels(ax_heatmap.get_yticklabels(), fontsize=20)

    #~~~~~~~~~~~~~~~~~~~~~ Subplot 5 - Relatório com Heatmap ~~~~~~~~~~~~~~~~~~~~~#
    ax_report = fig.add_subplot(gs[2,1])

    relatorio_formatado = gerar_relatorio_formatado(nome_input, chapa_input, media_presenca_semana, percentuais_presenca)

    ax_report.text(
        0, 1.058,
        relatorio_formatado,
        fontsize=20,
        ha='left',
        va='top',
        wrap=True,
        usetex=False
    )
    ax_report.axis('off')

    #~~~~~~~~~~~~~~~~~~~~~ Subplot 5.1 - Mini Heatmap Embutido ~~~~~~~~~~~~~~~~~~~~~#
    semanas_baixa = dias_presenca_por_semana[dias_presenca_por_semana < 2].index.tolist()
    tabela_heatmap_baixa = tabela_heatmap[
        tabela_heatmap.index.str.split(r" \|").str[0].isin(semanas_baixa)  # <- evita invalid escape sequence
    ]

    if not tabela_heatmap_baixa.empty:
        left = 0.69
        width = 0.30
        num_linhas = tabela_heatmap_baixa.shape[0]
        altura_base = 0.28
        linhas_base = 36
        height = (num_linhas / linhas_base) * altura_base
        top = 0.410
        bottom = top - height
        ax_heatmap3 = fig.add_axes([left, bottom, width, height])

        sns.heatmap(
            tabela_heatmap_baixa,
            cmap=ListedColormap(['#ffb3b3', '#f4cccc']),
            linewidths=1,
            linecolor='gray',
            annot=True,
            annot_kws={'size': 14},
            cbar=False,
            ax=ax_heatmap3
        )

        for y_idx, semana_label in enumerate(tabela_heatmap_baixa.index):
            semana_codigo = semana_label.split(" |")[0].replace(" ", "-")
            if semana_codigo in dias_presenca_por_semana.index:
                qtd_presenca = dias_presenca_por_semana.loc[semana_codigo]
                cor_destaque = "#ffb3b3" if qtd_presenca == 0 else "#f4cccc"
                for col_idx in range(tabela_heatmap_baixa.shape[1]):
                    ax_heatmap3.add_patch(
                        plt.Rectangle(
                            (col_idx, y_idx),
                            1, 1,
                            facecolor=cor_destaque,
                            edgecolor='gray',
                            linewidth=2,
                            zorder=1
                        )
                    )

        ax_heatmap3.set_ylabel('')
        ax_heatmap3.set_xlabel('')
        ax_heatmap3.set_xticklabels(ax_heatmap3.get_xticklabels(), fontsize=14)
        ax_heatmap3.set_yticklabels(ax_heatmap3.get_yticklabels(), fontsize=14, rotation=0)

    # ~~~~~~~~~~~~~~~~~~~~~ Subplot 6 - Heatmap Específico (linha "Total") ~~~~~~~~~~~~~~~~~~~~~
    ax_heatmap2 = fig.add_subplot(gs_heatmaps[1])

    # Filtra a pessoa
    df_chapa = df_count_por_semana[df_count_por_semana['Cadastro'].astype(str) == str(chapa_input)].copy()

    # Garante nomes de dias em PT (se já estiver em PT, mantém)
    df_chapa['Dia Semana'] = df_chapa['Dia Semana'].map(dias_portugues).fillna(df_chapa['Dia Semana'])

    # Totais por dia da semana; se não houver dados, reindex preenche com 0
    totais_dia = (df_chapa.groupby('Dia Semana')['Contagem Presença']
                .sum()
                .reindex(ordenacao_dias, fill_value=0))

    # DataFrame de 1 linha com índice "Total"
    heatmap_data = pd.DataFrame([totais_dia.values], columns=ordenacao_dias, index=['Total'])

    # Plota (usa cmap_personalizado já definido antes)
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt='g',
        cmap=cmap_personalizado,
        linewidths=1,
        linecolor='gray',
        annot_kws={'color': 'black', "size": 17},
        cbar=False,
        ax=ax_heatmap2
    )

    ax_heatmap2.set_ylabel('')
    ax_heatmap2.set_xlabel('')
    ax_heatmap2.tick_params(axis='y', left=True)
    ax_heatmap2.set_xticklabels(ax_heatmap2.get_xticklabels(), fontsize=20)
    ax_heatmap2.set_yticklabels(ax_heatmap2.get_yticklabels(), fontsize=20, rotation=0)

    # Ajusta layout para a exibição:
    plt.subplots_adjust(wspace=1.4)
    fig.tight_layout()
    img_base64 = fig_to_base64(fig)

    conteudo = html.Div([
        html.Button("Voltar", id='botao-voltar', n_clicks=0, style={
                'color': 'Black',
                'fontFamily': 'Aparajita',
                'fontWeight': 'bold',
                'marginTop': '20px',
                'display': 'block',
                'marginTop': '10px',
                'marginLeft': '10px',
                'marginRight': 'auto',
                'padding': '10px 20px',
                'fontSize': '14px',
                'backgroundColor': '#e0e0e0',
                'border': 'none',
                'borderRadius': '5px',
                'cursor': 'pointer',
                'boxShadow': '0px 2px 6px rgba(0, 0, 0, 0.25)'
            }),
        html.Img(src=img_base64, style={
            'width': '100%', 
            'maxWidth': '1400px', 
            'marginTop': '20px', 
            'background': 'white', 
            'display': 'block',
            'marginLeft': 'auto',
            'marginRight': 'auto'
            })
    ])

    return dcc.Loading(
        id="loading-individual",
        type="circle",
        fullscreen=True,
        children=conteudo
    ), {'display': 'block'}, {'display': 'none'}

def get_area(pathname):
    # /area/<slug>?dep=<slug>
    try:
        base = pathname.split("/area/")[1]
    except:
        return None, None
    if "?" in base:
        areaslug, query = base.split("?", 1)
        params = dict([p.split("=",1) for p in query.split("&") if "=" in p])
        dep = normalizar_str(params.get("dep")) if "dep" in params else None
    else:
        areaslug = base; dep = None
    area = normalizar_str(areaslug)
    return area, dep

def build_dataframe_diario_area(area, subarea, her_df, df_datas, df_dept):
    """
    Retorna:
      - df_full (DataFrame longo) com colunas: Data, Cadastro, Nome, Cargo, Area, Departamento, Presente (0/1)
      - datas (DatetimeIndex) com todo o período (min..max) filtrado
    """

    # ---- 1) Pessoas válidas pela combinação Área/Subárea + Herança (robusto) ----
    if subarea:
        mask_her = (her_df['Area'] == subarea)
    else:
        mask_her = her_df['Area'].astype(str).str.startswith(str(area))

    herancas_validas = her_df.loc[mask_her, 'Heranca'].dropna().unique()

    # Filtro base de área em departamentos
    if subarea:
        mask_area = (df_dept['Area'] == subarea)
    else:
        mask_area = df_dept['Area'].astype(str).str.startswith(str(area))

    # Só aplica filtro de herança se existir algo
    if len(herancas_validas) > 0 and 'Heranca' in df_dept.columns:
        mask_area = mask_area & (df_dept['Heranca'].isin(herancas_validas))

    base_dept = df_dept.loc[mask_area, ['Cadastro', 'Area', 'Departamento']].drop_duplicates()

    if base_dept.empty:
        # nada para plotar
        return pd.DataFrame(columns=['Data','Cadastro','Nome','Cargo','Area','Departamento','Presente']), pd.DatetimeIndex([])

    # ---- 2) Completar Nome/Cargo a partir do df_semana_merged ----
    ref_cols = ['Cadastro', 'Nome', 'Cargo']
    df_ref = df_semana_merged[ref_cols].drop_duplicates()
    chaves = base_dept.merge(df_ref, on='Cadastro', how='left')

    # ---- 3) Presenças diárias: filtra df_datas por Cadastro (não exige Nome/Cargo) ----
    dfp = df_datas[df_datas['Cadastro'].isin(chaves['Cadastro'])].copy()
    if dfp.empty:
        return pd.DataFrame(columns=['Data','Cadastro','Nome','Cargo','Area','Departamento','Presente']), pd.DatetimeIndex([])

    dfp['Data'] = pd.to_datetime(dfp['Data'])

    inicio, fim = dfp['Data'].min(), dfp['Data'].max()
    datas = pd.date_range(inicio, fim, freq='D')

    # ---- 4) Grade completa (todas as datas x todas as pessoas) ----
    grade = (
        pd.MultiIndex.from_product([datas, chaves['Cadastro'].unique()], names=['Data','Cadastro'])
          .to_frame(index=False)
          .merge(chaves, on='Cadastro', how='left')  # adiciona Area, Departamento, Nome, Cargo
    )

    # ---- 5) Marca 1 se teve presença no dia ----
    pres_diaria = (
        dfp.groupby(['Cadastro','Data'], as_index=False)
           .size()
           .rename(columns={'size':'Presente'})
    )
    pres_diaria['Presente'] = 1

    df_full = (
        grade.merge(pres_diaria, on=['Cadastro','Data'], how='left')
             .fillna({'Presente': 0})
    )
    df_full['Presente'] = df_full['Presente'].astype(int)

    # Saneamento mínimo:
    df_full['Data'] = pd.to_datetime(df_full['Data'])
    df_full['Presente'] = pd.to_numeric(df_full['Presente'], errors='coerce').fillna(0).astype(int)
    df_full = df_full.sort_values(['Cadastro', 'Data'])
    
    return df_full, datas

def grafico_linha_diaria_por_pessoa(df_full, datas, titulo):
    fig = go.Figure()
    if df_full.empty:
        fig.update_layout(title=titulo, height=500)
        return fig

    dfp = df_full.copy()
    dfp['Data'] = pd.to_datetime(dfp['Data'])
    dfp['Presente'] = pd.to_numeric(dfp['Presente'], errors='coerce').fillna(0).astype(int)

    # período total único p/ todas as pessoas (garante linha base contínua)
    min_d = dfp['Data'].min().normalize()
    max_d = dfp['Data'].max().normalize()
    full_range = pd.date_range(min_d, max_d, freq='D')

    # ordena pessoas: Mensalista (M) antes, depois por nome
    pessoas = (dfp[['Cadastro','Nome','Cargo']]
               .drop_duplicates()
               .sort_values(['Cargo','Nome'])
               .reset_index(drop=True))
    pessoas['idx'] = pessoas.index + 1

    for _, p in pessoas.iterrows():
        cad, nome, cargo, idx = p['Cadastro'], p['Nome'], p['Cargo'], int(p['idx'])

        s = (dfp[dfp['Cadastro']==cad][['Data','Presente']]
                .set_index('Data')
                .reindex(full_range)  # garante todos os dias
                .fillna(0).astype({'Presente': int}))
        s.index.name = 'Data'

        # 1) Linha base (ausência) contínua, cinza claro
        fig.add_trace(go.Scatter(
            x=full_range, y=[idx]*len(full_range),
            mode='lines',
            line=dict(width=2, color='#dddddd'),
            hoverinfo='skip',
            showlegend=False
        ))

        # 2) Traços curtinhos nos dias com presença (acima da base)
        pres = s.index[s['Presente'].eq(1)]
        if len(pres):
            xs, ys = [], []
            half = pd.Timedelta(hours=12)  # comprimento ~1 dia
            for d in pres:
                xs += [d - half, d + half, None]   # None separa os segmentos
                ys += [idx + 0.7, idx + 0.7, None]

            fig.add_trace(go.Scatter(
                x=xs, y=ys,
                mode='lines',
                line=dict(width=6, color=('#7cc3e9' if cargo=='M' else '#1574b3')),
                name=f"{cad} - {nome}",
                showlegend=False,
                hovertemplate=f"{cad} - {nome}<br>%{{x|%d/%m/%Y}}: Presente<extra></extra>"
            ))

    fig.update_layout(
        title=titulo,
        height=max(420, 24*len(pessoas)+180),
        margin=dict(l=30, r=30, t=60, b=30),
        plot_bgcolor='white',
        xaxis=dict(type='date', showgrid=False, zeroline=False),
        yaxis=dict(showticklabels=False, showgrid=False, zeroline=False,
                   range=[0, int(pessoas['idx'].max()) + 1.5])
    )
    
    return fig

def grafico_tendencia_m_vs_e(df_full, titulo):
    fig = go.Figure()
    if df_full.empty:
        fig.update_layout(title=titulo, height=400)
        return fig

    base = (df_full.assign(Data=pd.to_datetime(df_full['Data']),
                           Presente=pd.to_numeric(df_full['Presente'], errors='coerce').fillna(0).astype(float))
                  .groupby(['Data','Cargo'])['Presente']
                  .mean()
                  .reset_index())

    for cod, nome, cor in [('M','Mensalistas','#7cc3e9'), ('E','Executivos','#1574b3')]:
        s = base[base['Cargo']==cod].sort_values('Data')
        if s.empty:
            continue

        # Série principal
        fig.add_trace(go.Scatter(
            x=s['Data'], y=s['Presente'],
            mode='lines', name=nome,
            line=dict(width=3, color=cor),
            hovertemplate=f"{nome}<br>%{{x|%d/%m/%Y}}: %{{y:.2f}}<extra></extra>"
        ))

        # Tendência (ajuste no tempo real e volta como datetime)
        x_num = s['Data'].astype('int64') / 1e9  # segundos desde epoch
        y = s['Presente'].values.astype(float)

        if len(x_num) >= 2:
            deg = 1 if len(x_num) == 2 else 2
            coef = np.polyfit(x_num, y, deg)
            poly = np.poly1d(coef)
            x_pred = np.linspace(x_num.min(), x_num.max(), 300)
            y_pred = poly(x_pred)
            x_pred_dt = pd.to_datetime(x_pred, unit='s')

            cor_tend = '#256b2b' if float(y_pred[-1]) >= 2 else 'red'
            fig.add_trace(go.Scatter(
                x=x_pred_dt, y=y_pred,
                mode='lines',
                name=f"Tendência {nome}",
                line=dict(width=2, dash='dash', color=cor_tend),
                hoverinfo='skip',
                showlegend=True
            ))

    fig.update_layout(
        title=titulo,
        xaxis_title="Tempo (diário, comprimido)",
        yaxis_title="Média de Presença (dias)",
        height=420,
        legend=dict(bgcolor='rgba(255,255,255,0.7)'),
        plot_bgcolor='white'
    )
    fig.update_xaxes(showgrid=True, griddash='dot')
    fig.update_yaxes(showgrid=True, griddash='dot', range=[0, max(2.5, base['Presente'].max()+0.3)])
    return fig

@app.callback(
    Output('url', 'pathname'),
    Input('grafico-boxplot', 'clickData'),
    prevent_initial_call=True
)
def ir_para_area_via_boxplot(click):
    if not click or 'points' not in click or not click['points']:
        raise PreventUpdate
    area_nome = click['points'][0].get('label')
    if not area_nome:
        raise PreventUpdate
    return f"/area/{slug(area_nome)}"

@app.callback(
    Output('conteudo-analitico-area', 'children'),
    Output('conteudo-analitico-area', 'style'),
    Output('pagina-geral', 'style', allow_duplicate=True),
    Input('url', 'pathname'),
    Input('botao-voltar-area', 'n_clicks'),
    State('dropdown-area', 'value'),
    State('dropdown-subarea', 'value'),
    prevent_initial_call=True
)
def exibir_conteudo_area(pathname, n_clicks, area_atual, subarea_atual):
    if n_clicks and n_clicks > 0:
        return None, estilo_oculto, estilo_exibir

    if not pathname or '/area/' not in pathname:
        return None, estilo_oculto, estilo_exibir

    # área e dep vindos da URL; se ausentes, usa os dropdowns atuais
    area_url, dep_url = get_area(pathname)
    area = area_url or area_atual
    subarea = dep_url or subarea_atual

    if not area or area == 'GERAL':
        # nada para renderizar se não houver seleção válida
        return None, estilo_oculto, estilo_exibir

    # monta dataframe diário binário (0/1) e gráficos
    df_full, datas = build_dataframe_diario_area(area, subarea, her, df_datas, departamentos)
    titulo_area = f"Linhas Diárias por Pessoa"
    fig_pessoas = grafico_linha_diaria_por_pessoa(df_full, datas, titulo_area)

    titulo_tend = f"Tendência Diária: Mensalistas x Executivos"
    fig_tend = grafico_tendencia_m_vs_e(df_full, titulo_tend)

    conteudo = html.Div([
        html.Button("Voltar", id='botao-voltar-area', n_clicks=0, style={
            'color':'Black','fontFamily':'Aparajita','fontWeight':'bold',
            'marginTop':'20px','display':'block','marginLeft':'10px',
            'marginRight':'auto','padding':'10px 20px','fontSize':'14px',
            'backgroundColor':'#e0e0e0','border':'none','borderRadius':'5px',
            'cursor':'pointer','boxShadow':'0px 2px 6px rgba(0,0,0,0.25)'
        }),
        html.H2(f"Análise Específica — {subarea or area}", style={'textAlign':'left','marginLeft':'10px'}),
        dcc.Graph(figure=fig_pessoas, id='grafico-linha-diaria-pessoas', style={'height':'auto','minHeight':'400px'}),
        dcc.Graph(figure=fig_tend, id='grafico-tendencia-m-e', style={'height':'460px'})
    ], style={'padding':'10px','backgroundColor':'white'})

    return conteudo, estilo_exibir, estilo_oculto

if __name__ == '__main__':
    app.run(port=8080, debug=False)

#### Extremos de cada caso (Home Office e Presencial): Criado pelo Nicholas Birochi

In [ ]:
'''# Filtra df_semana_merged para a chapa informada e somente registros com Contagem Presença menor que '2':
df_filtrado_presenca = df_ronda[
    (df_ronda['Cadastro'] == '1730037') & 
    (df_ronda['Data'] == '30/04/2024')
]

df_filtrado_presenca = df_filtrado_presenca[['Cadastro', 'Nome', 'Dir', 'Tipo',	'Acesso', 'Planta',	'Coletor', 'Descricao Coletor',	'Uso Acesso', 'Data', 'Hora']]

# Exibe os resultados de presenças filtradas:
print("Distribuição Presenças:")
display(df_filtrado_presenca)'''

In [ ]:
'''# Se a chapa foi informada, filtra os DataFrames:
chapa_input = 4024283
if chapa_input:
    # Filtra df_semana_merged para a chapa informada e somente registros com Contagem Presença menor que '2':
    df_filtrado_presenca = df_semana_merged[
        (df_semana_merged['Cadastro'] == chapa_input) & 
        (df_semana_merged['Contagem Presença'] < 2)
    ]
    
    # Filtra df_homeoffice_semana_merged para a chapa informada e somente registros com Contagem Home Office maior que '3':
    df_filtrado_homeoffice = df_homeoffice_semana_merged[
        (df_homeoffice_semana_merged['Cadastro'] == chapa_input) & 
        (df_homeoffice_semana_merged['Contagem Home Office'] > 3)
    ]
else:
    # Caso nenhuma chapa tenha sido informada, considera todos os dados:
    df_filtrado_presenca = df_semana_merged
    df_filtrado_homeoffice = df_homeoffice_semana_merged

# Se o DataFrame de presenças não estiver vazio:
if not df_filtrado_presenca.empty:
    # Cria uma cópia para evitar SettingWithCopyWarning:
    df_filtrado_presenca = df_filtrado_presenca.copy()
    # Formata datas para string no formato 'dd/mm/aaaa':
    df_filtrado_presenca['Início da Semana'] = df_filtrado_presenca['Início da Semana'].dt.strftime('%d/%m/%Y')
    df_filtrado_presenca['Fim da Semana'] = df_filtrado_presenca['Fim da Semana'].dt.strftime('%d/%m/%Y')

# Se o DataFrame de home office não estiver vazio:
if not df_filtrado_homeoffice.empty:
    # Cria uma cópia para evitar SettingWithCopyWarning:
    df_filtrado_homeoffice = df_filtrado_homeoffice.copy()
    # Formata datas para string no formato 'dd/mm/aaaa':
    df_filtrado_homeoffice['Início da Semana'] = df_filtrado_homeoffice['Início da Semana'].dt.strftime('%d/%m/%Y')
    df_filtrado_homeoffice['Fim da Semana'] = df_filtrado_homeoffice['Fim da Semana'].dt.strftime('%d/%m/%Y')

# Seleciona somente as colunas desejadas no DataFrame de presenças:
df_filtrado_presenca = df_filtrado_presenca[
    ['Cadastro', 'Contagem Presença', 'Início da Semana', 'Fim da Semana', 'Ano', 'Semana por Ano']
]

# Seleciona somente as colunas desejadas no DataFrame de home office:
df_filtrado_homeoffice = df_filtrado_homeoffice[
    ['Cadastro', 'Contagem Home Office', 'Início da Semana', 'Fim da Semana', 'Ano', 'Semana por Ano']
]

# Exibe os resultados de presenças filtradas:
print("Distribuição Presenças:")
display(df_filtrado_presenca)

# Exibe os resultados de home office filtrados:
print("\n\nDistribuição Home Office:")
display(df_filtrado_homeoffice)'''

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

# Análise Futura: Criado pelo Nicholas Birochi

In [ ]:
'''# Identificando pessoas que não sairam da empresa ou que fecharam o ponto em casa:
df_presencial_filtrado'''

In [ ]:
'''# Filtra só as linhas onde Dir == 'E' e se é 'Mensalista':
df_e = df_presencial_filtrado[(df_presencial_filtrado['Dir'] == 'E') & (df_presencial_filtrado['Cargo'] == 'M')]

# Conta quantos 'E' cada pessoa tem por data:
contagem_e = df_e.groupby(['Data', 'Nome']).size()

# Filtra os pares (Data, Nome) com mais de 2 ocorrências:
pares_validos_e = contagem_e[contagem_e > 2].index

# Filtra 'df_e' para manter só as linhas desses pares:
df_filtrado_e = df_e[df_e.set_index(['Data', 'Nome']).index.isin(pares_validos_e)]

# Visualizando o resultado:
df_filtrado_e'''

In [ ]:
'''# Filtra só as linhas onde Dir == 'S' e se é 'Mensalista':
df_s = df_presencial_filtrado[(df_presencial_filtrado['Dir'] == 'S') & (df_presencial_filtrado['Cargo'] == 'M')]

# Conta quantos 'S' cada pessoa tem por data:
contagem_s = df_s.groupby(['Data', 'Nome']).size()

# Filtra os pares (Data, Nome) com mais de 2 ocorrências:
pares_validos_s = contagem_s[contagem_s > 2].index

# Filtra 'df_s'para manter só as linhas desses pares:
df_filtrado_s = df_s[df_s.set_index(['Data', 'Nome']).index.isin(pares_validos_s)]

# Visualizando o resultado:
df_filtrado_s'''

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

# Validando as informações da Base de Dados: Criado pelo Nicholas Birochi

In [ ]:
'''# Função para classificar a semana em um trimestre personalizado:
def classificar_trimestre(semana):
    if 2 <= semana <= 13:
        return 1
    elif 14 <= semana <= 26:
        return 2
    elif 27 <= semana <= 39:
        return 3
    elif 40 <= semana <= 52:
        return 4
    else:
        return None

# Lista para armazenar os registros de férias prolongadas:
ferias_prolongadas_geral = []

# Percorre todas as chapas no DataFrame original:
for chapa, grupo in df_semana_merged.groupby('Cadastro'):
    grupo = grupo.sort_values(by=['Ano', 'Semana por Ano']).reset_index(drop=True)
    grupo['Ausente'] = grupo['Contagem Presença'].apply(lambda x: 1 if x == 0 else 0)
    grupo['Grupo'] = (grupo['Ausente'] != grupo['Ausente'].shift()).cumsum()

    # Agrupa por blocos consecutivos de ausência:
    blocos_ausencia = grupo[grupo['Ausente'] == 1].groupby('Grupo')

    # Filtra blocos com 4 semanas ou mais de ausência:
    for _, bloco in blocos_ausencia:
        if len(bloco) >= 4:
            nome = bloco['Nome'].iloc[0].title()
            semana_inicio = int(bloco['Semana por Ano'].iloc[0])
            semana_fim = int(bloco['Semana por Ano'].iloc[-1])
            ano = int(bloco['Ano'].iloc[0])
            semanas_ausente = len(bloco)
            trimestre = classificar_trimestre(semana_inicio)

            if trimestre is not None:
                ferias_prolongadas_geral.append({
                    'Cadastro': chapa,
                    'Nome': nome,
                    'Ano': ano,
                    'Semana Início': semana_inicio,
                    'Semana Fim': semana_fim,
                    'Semanas Ausente': semanas_ausente,
                    'Trimestre': trimestre,
                    'Trimestre_Label': f"{ano} - T{trimestre}"
                })

# Transforma a lista em DataFrame:
df_ferias_prolongadas = pd.DataFrame(ferias_prolongadas_geral)

# Agrupa os dados por colaborador e trimestre:
df_trimestres = (
    df_ferias_prolongadas
    .groupby(['Cadastro', 'Nome', 'Ano', 'Trimestre_Label'])
    .agg({
        'Semanas Ausente': 'sum',
        'Semana Início': 'min',
        'Semana Fim': 'max'
    })
    .reset_index()
    .sort_values(['Trimestre_Label', 'Semanas Ausente'], ascending=[True, False])
)

# Cria coluna auxiliar apenas com o número do trimestre (ex: T1, T2, T3, T4):
df_trimestres['Trimestre_Num'] = df_trimestres['Trimestre_Label'].str.extract(r'T(\d)').astype(int)

# Conta quantos trimestres distintos cada colaborador teve ausência prolongada:
trimestres_por_cadastro = df_trimestres.groupby('Cadastro')['Trimestre_Num'].nunique()

# Pega chapas que apareceram em mais de um trimestre:
chapas_todos_trimestres = trimestres_por_cadastro[trimestres_por_cadastro > 1].index

# Remove esses colaboradores do DataFrame:
df_trimestres = df_trimestres[~df_trimestres['Cadastro'].isin(chapas_todos_trimestres)]

# Filtra pelo número do trimestre:
df_trimestres = df_trimestres[df_trimestres['Trimestre_Num'] == 4]

# Exibe resultado final pós o filtro:
df_trimestres'''

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

# Salvando toda a Base de Dados para usar no aplicativo: Criado pelo Nicholas Birochi

In [ ]:
# Visualizando a Base de Dados antes do tratamento:
df_ronda.head()

In [ ]:
# Variável:
dias_semana = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado', 'Domingo']

# Converter 'Data' para datetime:
df_ronda['Data'] = pd.to_datetime(df_ronda['Data'], format='%d/%m/%Y', errors='coerce')

# Manter apenas colunas relevantes (base original de registros de presença):
cols_base = ['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Planta',
             'Cargo', 'Cargo Especifico', 'Afastado', 'Data']
df_ronda_base = df_ronda[cols_base].dropna(subset=['Cadastro', 'Data']).copy()

# Definir períodos de férias coletivas:
ferias = [
    (datetime(2023, 12, 18), datetime(2024, 1, 7)),
    (datetime(2024, 12, 23), datetime(2025, 1, 12)),
]

# Índice de datas de férias (mais rápido para filtrar do que checar uma-a-uma)
ferias_idx = pd.DatetimeIndex([])
for inicio, fim in ferias:
    ferias_idx = ferias_idx.union(pd.date_range(inicio, fim, freq='D'))

# Pessoas únicas (traz os metadados de cada colaborador)
pessoas = df_ronda_base[['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Planta',
                         'Cargo', 'Cargo Especifico', 'Afastado']].drop_duplicates()

# Período (min/max) de atividade observado na base para cada pessoa
periodos = (df_ronda_base.groupby('Cadastro')['Data']
            .agg(inicio='min', fim='max')
            .reset_index())

# Gera todas as datas úteis por pessoa dentro do período observado (exclui férias coletivas):
# Obs.: usa dias corridos (freq='D'); se quiser só úteis, troque para freq='B'...
linhas = []
for _, r in periodos.iterrows():
    rng = pd.date_range(r['inicio'], r['fim'], freq='D')
    # remove férias coletivas
    rng = pd.DatetimeIndex(rng).difference(ferias_idx)
    if len(rng):
        linhas.append(pd.DataFrame({'Cadastro': r['Cadastro'], 'Data': rng}))

todas_datas_por_pessoa = pd.concat(linhas, ignore_index=True) if linhas else pd.DataFrame(columns=['Cadastro', 'Data'])

# Produto "controlado": pessoas × datas (apenas no período da própria pessoa):
combinacoes = pessoas.merge(todas_datas_por_pessoa, on='Cadastro', how='inner')

# Presenças reais (1) por cadastro/data:
presencas = (df_ronda_base[['Cadastro', 'Data']]
             .drop_duplicates()
             .assign(**{'Contagem Presença': 1}))

# Junta combinações com presenças:
df_ronda = combinacoes.merge(presencas, on=['Cadastro', 'Data'], how='left')

# Ausências -> 0:
df_ronda['Contagem Presença'] = df_ronda['Contagem Presença'].fillna(0).astype(int)

# Ano/Semana ISO:
iso = df_ronda['Data'].dt.isocalendar()
df_ronda['Ano'] = iso.year.astype(int)
df_ronda['Semana por Ano'] = iso.week.astype(int)

# Semana_Ano (ex: "2024-31"):
df_ronda['Semana_Ano'] = df_ronda['Ano'].astype(str) + '-' + df_ronda['Semana por Ano'].astype(str).str.zfill(2)

# Dia da semana (0=Segunda ... 6=Domingo):
df_ronda['Dia Semana'] = df_ronda['Data'].dt.dayofweek.map(lambda x: dias_semana[int(x)])

# Ordenar resultado
df_ronda = df_ronda.sort_values(['Cadastro', 'Data']).reset_index(drop=True)

# Ordenar colunas na ordem desejada:
df_ronda = df_ronda[['Cadastro', 'Nome', 'Area', 'Departamento', 'Heranca', 'Planta',
                     'Cargo', 'Cargo Especifico', 'Afastado', 'Data', 'Ano', 'Semana_Ano',
                     'Semana por Ano', 'Dia Semana', 'Contagem Presença']]

# Removendo duplicatas:
df_ronda = df_ronda.sort_values(['Cadastro','Data']).drop_duplicates(['Cadastro','Data'], keep='last').reset_index(drop=True)

# Visualizando a Base de Dados depois do tratamento final:
df_ronda

In [ ]:
# Exportando os Dados Originais:
df_semana_merged.to_parquet(R"..\..\..\Base de Dados Original\ArquivoFinal.parquet")
df_presenca_merged_por_cadastro.to_parquet(R"..\..\..\Base de Dados Original\ArquivoFinalPessoa.parquet")
df_datas.to_parquet(R"..\..\..\Base de Dados Original\ArquivoFinalDatas.parquet")
afastados.to_parquet(R"..\..\..\Base de Dados Original\Afastados.parquet")
departamentos.to_parquet(R"..\..\..\Base de Dados Original\Departamentos.parquet")
gptw.to_parquet(R"..\..\..\Base de Dados Original\GPTW.parquet")